In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
#mzk preprocess

# import re
# import pandas as pd
# from snowflake.snowpark.functions import upper, col
# import numpy as np

# def clean_muzooka_artist_name(name):
#     """
#     Cleans a Muzooka artist name by removing special characters, normalizing spaces,
#     and handling common stop words.
#     """
#     if name is None or pd.isna(name):
#         return ""

#     try:
#         name = str(name).upper()
#         name_no_apostrophes = re.sub(r'[\'"-]', '', name)
#         name_with_spaces = re.sub(r'["`_&+\]\[]', '', name_no_apostrophes)
#         name_with_spaces = name_with_spaces.replace('-', ' ')
#         name_with_spaces = name_with_spaces.replace('"', '')

#         cleaned = re.sub(r'[^a-zA-Z0-9 ]', '', name_with_spaces)
#         cleaned = re.sub(r'\s+', ' ', cleaned).strip()

#         if cleaned == "":
#             return name

#         stop_words = [r'\bA\b', r'\bAN\b', r'\bTHE\b']

#         for word in stop_words:
#             cleaned = re.sub(word, ' ', cleaned)

#         cleaned = re.sub(r'\s+', ' ', cleaned).strip()

#         if cleaned == "":
#             return name_with_spaces.strip()

#         return cleaned
#     except:
#         return ""

# def normalize_special_chars_optimized(text):
#     """
#     Optimized version to normalize special characters in Muzooka artist names.
#     """
#     if not text:
#         return ""

#     try:
#         text = str(text).upper()
#         text = re.sub(r'^[\[\(\{]', '', text)
#         text = re.sub(r'[\'"`\-]', '', text)

#         chars_map = {
#             'Ø': 'O', 'Ö': 'O', 'Ó': 'O', 'Ò': 'O', 'Ô': 'O', 'Õ': 'O',
#             'Ä': 'A', 'Á': 'A', 'À': 'A', 'Â': 'A', 'Ã': 'A',
#             'Ë': 'E', 'É': 'E', 'È': 'E', 'Ê': 'E',
#             'Ü': 'U', 'Ú': 'U', 'Ù': 'U', 'Û': 'U',
#             'Ï': 'I', 'Í': 'I', 'Ì': 'I', 'Î': 'I',
#             'Ñ': 'N', 'Ç': 'C'
#         }
#         for special, basic in chars_map.items():
#             text = text.replace(special, basic)

#         return text
#     except:
#         return ""

# def balance_brackets(text):
#     """
#     Balances brackets in a string by adding missing closing brackets or removing extra closing brackets.
#     Handles three types of brackets: (), [], {}.
#     """
#     if not text:
#         return ""

#     bracket_pairs = {
#         '(': ')',
#         '[': ']',
#         '{': '}'
#     }

#     opening_brackets = set(bracket_pairs.keys())
#     closing_brackets = set(bracket_pairs.values())

#     stack = []
#     result = list(text)

#     i = 0
#     while i < len(result):
#         char = result[i]

#         if char in opening_brackets:
#             stack.append((char, i))
#         elif char in closing_brackets:
#             if not stack:
#                 result[i] = ''
#                 i += 1
#                 continue
#             else:
#                 last_open, _ = stack[-1]
#                 expected_close = bracket_pairs[last_open]

#                 if char == expected_close:
#                     stack.pop()
#                 else:
#                     result[i] = expected_close
#                     stack.pop()
#         i += 1

#     while stack:
#         open_bracket, _ = stack.pop()
#         result.append(bracket_pairs[open_bracket])

#     return ''.join(result)

# def split_muzooka_artist_name(name):
#     """
#     Splits a Muzooka artist name into its constituent parts based on delimiters and bracketed content.
#     """
#     if not name:
#         return []

#     try:
#         original_name = str(name).upper() if name is not None else ""
#         name_normalized = original_name.replace("'", "").replace("?", "")
#         name_normalized = balance_brackets(name_normalized)
#     except:
#         return []

#     all_parts = []
#     bracket_contents = []
#     bracket_pairs = [
#         ('(', ')'),
#         ('[', ']'),
#         ('{', '}')
#     ]

#     working_name = name_normalized

#     # Artist-specific delimiters
#     word_delimiters = [
#         ' FT ', ' FT. ', ' FEAT ', ' FEAT. ', ' WITH ', ' AND ',
#         ' VS ', ' VS. ', ' FEATURING ', ' EN ',
#         ' AKA ', ' A.K.A. ', ' AKA. '
#     ]

#     for open_bracket, close_bracket in bracket_pairs:
#         start_pos = working_name.find(open_bracket)
#         while start_pos != -1:
#             end_pos = working_name.find(close_bracket, start_pos + 1)
#             if end_pos != -1:
#                 bracket_content = working_name[start_pos + 1:end_pos].strip()
#                 if bracket_content:
#                     bracket_contents.append(bracket_content)
#                 working_name = working_name[:start_pos] + working_name[end_pos + 1:]
#             else:
#                 start_pos = working_name.find(open_bracket, start_pos + 1)
#                 continue
#             start_pos = working_name.find(open_bracket)

#     main_part = working_name.strip()
#     main_parts = []
#     has_explicit_delimiters = False

#     if main_part:
#         delimiters = ['|', '/', '#', '\\', ',', ';', '`', '_', '&', '+']
#         main_with_boundaries = ' ' + main_part + ' '

#         for d in delimiters:
#             if d in main_with_boundaries:
#                 has_explicit_delimiters = True
#                 break

#         if not has_explicit_delimiters:
#             for wd in word_delimiters:
#                 if wd in ' ' + main_with_boundaries + ' ':
#                     has_explicit_delimiters = True
#                     break

#         for d in delimiters:
#             main_with_boundaries = main_with_boundaries.replace(d, '|')

#         for wd in word_delimiters:
#             if wd in main_with_boundaries:
#                 main_with_boundaries = main_with_boundaries.replace(wd, '|')

#         main_parts = [part.strip() for part in main_with_boundaries.split('|') if part.strip()]
#         for part in main_parts:
#             clean_part = normalize_special_chars_optimized(part)
#             if clean_part and len(clean_part) > 1:
#                 all_parts.append(clean_part)

#     for content in bracket_contents:
#         delimiter_word_at_start = None
#         for delimiter_word in ['WITH', 'FEAT', 'FEAT.', 'FT', 'FT.', 'AND', 'FEATURING']:
#             if content.startswith(delimiter_word + ' '):
#                 delimiter_word_at_start = delimiter_word
#                 break

#         if delimiter_word_at_start:
#             rest_of_content = content[len(delimiter_word_at_start):].strip()
#             has_more_delimiters = False

#             for d in ['|', '/', ',', '&', '+', '-']:
#                 if d in rest_of_content:
#                     has_more_delimiters = True
#                     break

#             if not has_more_delimiters:
#                 for wd in [' AND ', ' WITH ', ' FT ', ' FT. ', ' FEAT ', ' FEAT. ']:
#                     if wd in f" {rest_of_content} ":
#                         has_more_delimiters = True
#                         break

#             if has_more_delimiters:
#                 cleaned = rest_of_content
#                 for d in ['|', '/', ',', '&', '+', '-']:
#                     cleaned = cleaned.replace(d, '|')

#                 for wd in [' AND ', ' WITH ', ' FT ', ' FT. ', ' FEAT ', ' FEAT. ']:
#                     if wd in f" {cleaned} ":
#                         cleaned = cleaned.replace(wd.strip(), '|')

#                 for part in cleaned.split('|'):
#                     part = part.strip()
#                     if part:
#                         clean_part = normalize_special_chars_optimized(part)
#                         if clean_part and len(clean_part) > 1:
#                             all_parts.append(clean_part)
#             else:
#                 clean_part = normalize_special_chars_optimized(rest_of_content)
#                 if clean_part and len(clean_part) > 1:
#                     all_parts.append(clean_part)
#         else:
#             has_delimiter = False

#             for d in ['|', '/', ',', '&', '+', '-']:
#                 if d in content:
#                     has_delimiter = True
#                     break

#             if not has_delimiter:
#                 for wd in [' AND ', ' WITH ', ' FT ', ' FT. ', ' FEAT ', ' FEAT. ']:
#                     if wd in f" {content} ":
#                         has_delimiter = True
#                         break

#             if has_delimiter:
#                 cleaned = content
#                 for d in ['|', '/', ',', '&', '+', '-']:
#                     cleaned = cleaned.replace(d, '|')

#                 for wd in [' AND ', ' WITH ', ' FT ', ' FT. ', ' FEAT ', ' FEAT. ']:
#                     if wd in f" {cleaned} ":
#                         cleaned = cleaned.replace(wd.strip(), '|')

#                 for part in cleaned.split('|'):
#                     part = part.strip()
#                     if part:
#                         clean_part = normalize_special_chars_optimized(part)
#                         if clean_part and len(clean_part) > 1:
#                             all_parts.append(clean_part)
#             else:
#                 clean_part = normalize_special_chars_optimized(content)
#                 if clean_part and len(clean_part) > 1:
#                     all_parts.append(clean_part)

#     if len(main_parts) == 2 and not has_explicit_delimiters:
#         reversed_name = f"{main_parts[1]} {main_parts[0]}"
#         clean_reversed = normalize_special_chars_optimized(reversed_name)
#         if clean_reversed and clean_reversed != normalize_special_chars_optimized(main_part):
#             all_parts.append(clean_reversed)

#     if len(all_parts) == 0 and main_part:
#         clean_full_name = normalize_special_chars_optimized(main_part)
#         if clean_full_name and len(clean_full_name) > 1:
#             all_parts.append(clean_full_name)

#     if len(main_parts) == 1 and main_part:
#         clean_full_name = normalize_special_chars_optimized(main_part)
#         if clean_full_name and len(clean_full_name) > 1:
#             if clean_full_name not in all_parts:
#                 all_parts.append(clean_full_name)

#     unique_parts = []
#     seen = set()
#     for part in all_parts:
#         clean_part = part.replace('(', '').replace(')', '').replace('?', '')
#         if clean_part and clean_part not in seen and len(clean_part) > 1:
#             seen.add(clean_part)
#             unique_parts.append(clean_part)

#     return unique_parts


# def preprocess_muzooka_artist_dataframe_vectorized(df, name_col='MUZOOKA_ARTIST_NAME', id_col='RECORDINGS_ID', batch_size=10000):
#     """
#     Vectorized preprocessing of Muzooka artist data using pandas operations.
#     """
#     df_pandas = df.to_pandas()
#     total_rows = len(df_pandas)

#     print(f"Processing {total_rows} rows in vectorized mode...")

#     df_pandas[name_col] = df_pandas[name_col].fillna("")

#     print("Applying clean_muzooka_artist_name...")
#     df_pandas['CLEAN_NAME'] = df_pandas[name_col].apply(clean_muzooka_artist_name)

#     print("Applying normalize_special_chars_optimized...")
#     df_pandas['NORMALIZED_NAME'] = df_pandas[name_col].apply(normalize_special_chars_optimized)

#     print("Applying split_muzooka_artist_name...")
#     df_pandas['DELIMITED_PARTS'] = df_pandas[name_col].apply(split_muzooka_artist_name)

#     result_df = df.session.create_dataframe(df_pandas)

#     return result_df


# def create_expanded_muzooka_artist_dataframe_vectorized(df, name_col='MUZOOKA_ARTIST_NAME', id_col='RECORDINGS_ID', batch_size=10000):
#     """
#     Vectorized expansion of Muzooka artist dataframe with artist variants.
#     """
#     df_pandas = df.to_pandas()
#     total_rows = len(df_pandas)

#     print(f"Expanding {total_rows} rows in vectorized mode...")

#     expanded_rows = []

#     for idx, row in df_pandas.iterrows():
#         original_row = row.to_dict()
#         original_row['IS_VARIANT'] = 'ORIGINAL'
#         expanded_rows.append(original_row)

#         if 'DELIMITED_PARTS' not in row or not row['DELIMITED_PARTS']:
#             continue

#         original_name_upper = ""
#         if row[name_col] is not None:
#             original_name_upper = str(row[name_col]).upper().replace("'", "")

#         for part in row['DELIMITED_PARTS']:
#             if part and part != original_name_upper and len(part) > 2:
#                 variant_row = row.to_dict()
#                 variant_row[name_col] = part
#                 variant_row['CLEAN_NAME'] = clean_muzooka_artist_name(part)
#                 variant_row['NORMALIZED_NAME'] = normalize_special_chars_optimized(part)
#                 variant_row['IS_VARIANT'] = 'VARIANT'
#                 variant_row['DELIMITED_PARTS'] = []
#                 expanded_rows.append(variant_row)

#     expanded_df = pd.DataFrame(expanded_rows)
#     result_df = df.session.create_dataframe(expanded_df)

#     return result_df


# def process_muzooka_artists_optimized(session, mode="full"):
#     """
#     Main function to process Muzooka artists with performance improvements.
#     """
#     print(f"=== OPTIMIZED MUZOOKA ARTIST PROCESSING ===")
#     print(f"Mode: {mode} (NOTE: Muzooka Artist data is ALWAYS processed in FULL)")

#     if mode == "example":
#         print("Creating example Muzooka artist data...")
#         muzooka_artist_examples = pd.DataFrame({
#             'RECORDINGS_ID': [1, 2, 3, 4, 5, 6, 7],
#             'MUZOOKA_TRACK_ID': ['TRACK_A', 'TRACK_B', 'TRACK_C', 'TRACK_D', 'TRACK_E', 'TRACK_F', 'TRACK_G'],
#             'MUZOOKA_ARTIST_NAME': [
#                 'ED SHEERAN',
#                 'ARCTIC MONKEYS (LIVE AT GLASTONBURY)',
#                 'QUEEN + ADAM LAMBERT',
#                 'POST MALONE FT. SZA',
#                 'COLDPLAY / BEYONCE',
#                 'IMAGINE DRAGONS A.K.A. DRAGONS',
#                 'DUA LIPA & ANNE-MARIE'
#             ]
#         })

#         muzooka_artist_df = session.create_dataframe(muzooka_artist_examples)

#         print("\nPreprocessing Muzooka artist data (VECTORIZED)...")
#         muzooka_artist_preprocessed = preprocess_muzooka_artist_dataframe_vectorized(muzooka_artist_df, name_col='MUZOOKA_ARTIST_NAME', id_col='RECORDINGS_ID')

#         print("\nCreating expanded Muzooka artist dataframe (VECTORIZED)...")
#         muzooka_artist_expanded = create_expanded_muzooka_artist_dataframe_vectorized(muzooka_artist_preprocessed, name_col='MUZOOKA_ARTIST_NAME', id_col='RECORDINGS_ID')

#         print("\nPreprocessed Muzooka Artist Data:")
#         muzooka_artist_preprocessed_pandas = muzooka_artist_preprocessed.to_pandas()
#         for _, row in muzooka_artist_preprocessed_pandas.iterrows():
#             print(f"RECORDINGS_ID: {row['RECORDINGS_ID']}, MUZOOKA_TRACK_ID: {row['MUZOOKA_TRACK_ID']}, Name: {row['MUZOOKA_ARTIST_NAME']}")
#             print(f"  Clean Name: {row['CLEAN_NAME']}")
#             print(f"  Normalized Name: {row['NORMALIZED_NAME']}")
#             print(f"  Delimited Parts: {row['DELIMITED_PARTS']}")
#             print()

#         print("\nExpanded Muzooka Artist Data:")
#         muzooka_artist_expanded_pandas = muzooka_artist_expanded.to_pandas()
#         for _, row in muzooka_artist_expanded_pandas.iterrows():
#             print(f"RECORDINGS_ID: {row['RECORDINGS_ID']}, MUZOOKA_TRACK_ID: {row['MUZOOKA_TRACK_ID']}, Name: {row['MUZOOKA_ARTIST_NAME']}, Variant: {row['IS_VARIANT']}")
#             print(f"  Clean Name: {row['CLEAN_NAME']}")
#             print()

#         return muzooka_artist_expanded

#     else:
#         print("Processing FULL Muzooka artist dataset with optimizations...")

#         muzooka_artist_table = "EDW_APPS.MATCHING.MZK_TRACKS_MATCHED_ARTISTS"
#         muzooka_artist_expanded_table = "EDW_APPS.MATCHING.MZK_TRACKS_ARTISTS_EXPANDED_CLEAN" # Or desired output table name
#         muzooka_artist_preprocessed_table = "EDW_APPS.MATCHING.MZK_ARTISTS_PREPROCESSED_TEMP"

#         try:
#             muzooka_artist_query = f"SELECT * FROM {muzooka_artist_table}"

#             print(f"Executing query: {muzooka_artist_query}")
#             muzooka_artist_df = session.sql(muzooka_artist_query)
#             muzooka_artist_count = muzooka_artist_df.count()
#             print(f"Muzooka Artist FULL dataset contains {muzooka_artist_count} rows")

#             if muzooka_artist_count == 0:
#                 print("Warning: Muzooka Artist DataFrame is empty.")
#                 return None

#             # Step 1: Preprocess Muzooka data using vectorized operations
#             print("\nPreprocessing Muzooka artist data (VECTORIZED FULL DATASET)...")
#             muzooka_artist_preprocessed = preprocess_muzooka_artist_dataframe_vectorized(muzooka_artist_df, name_col='MUZOOKA_ARTIST_NAME', id_col='RECORDINGS_ID', batch_size=20000)

#             # Save intermediate results
#             print(f"Saving preprocessed Muzooka artist data to {muzooka_artist_preprocessed_table}...")
#             muzooka_artist_preprocessed.write.mode("overwrite").save_as_table(muzooka_artist_preprocessed_table)

#             # Step 2: Create expanded Muzooka artist dataframe using vectorized operations
#             print("\nCreating expanded Muzooka artist dataframe (VECTORIZED FULL DATASET)...")
#             muzooka_artist_preprocessed = session.table(muzooka_artist_preprocessed_table)
#             muzooka_artist_expanded = create_expanded_muzooka_artist_dataframe_vectorized(muzooka_artist_preprocessed, name_col='MUZOOKA_ARTIST_NAME', id_col='RECORDINGS_ID', batch_size=20000)

#             # Save final Muzooka artist results
#             print(f"Saving expanded Muzooka data to {muzooka_artist_expanded_table}...")
#             muzooka_artist_expanded.write.mode("overwrite").save_as_table(muzooka_artist_expanded_table)

#             # Clean up intermediate tables
#             print("\nCleaning up intermediate tables...")
#             session.sql(f"DROP TABLE IF EXISTS {muzooka_artist_preprocessed_table}").collect()

#             print("Optimized Muzooka artist preprocessing completed successfully!")
#             print(f"Final results: {muzooka_artist_count} Muzooka artists processed")
#             print(f"Output table: {muzooka_artist_expanded_table}")

#             return muzooka_artist_expanded

#         except Exception as e:
#             print(f"Error during Muzooka artist processing: {str(e)}")
#             print("Stack trace:")
#             import traceback
#             traceback.print_exc()
#             return None

# if __name__ == "__main__":
#     result = process_muzooka_artists_optimized(session, mode="full")

In [ ]:
#adc preprocess


# import re
# import pandas as pd
# from snowflake.snowpark.functions import upper, col
# import numpy as np
# import string

# # --- Pre-compile all regex patterns for significant speed improvement ---
# APOSTROPHE_PATTERN = re.compile(r'[\'"-]')
# SPECIAL_CHARS_PATTERN = re.compile(r'["`_&+\]\[]')
# HYPHEN_PATTERN = re.compile(r'-')
# QUOTE_PATTERN = re.compile(r'"')
# NON_ALPHANUM_PATTERN = re.compile(r'[^a-zA-Z0-9 ]')
# MULTIPLE_SPACES_PATTERN = re.compile(r'\s+')
# STOP_WORDS_PATTERNS = [re.compile(pattern) for pattern in [r'\bA\b', r'\bAN\b', r'\bTHE\b']]

# # Pre-compile patterns for normalize_special_chars_optimized
# LEADING_BRACKETS_PATTERN = re.compile(r'^[\[\(\{]')
# APOSTROPHE_HYPHEN_PATTERN = re.compile(r'[\'"`\-]')

# # Character translation table for faster character replacement
# ACCENT_TRANSLATION = str.maketrans({
#     'Ø': 'O', 'Ö': 'O', 'Ó': 'O', 'Ò': 'O', 'Ô': 'O', 'Õ': 'O',
#     'Ä': 'A', 'Á': 'A', 'À': 'A', 'Â': 'A', 'Ã': 'A',
#     'Ë': 'E', 'É': 'E', 'È': 'E', 'Ê': 'E',
#     'Ü': 'U', 'Ú': 'U', 'Ù': 'U', 'Û': 'U',
#     'Ï': 'I', 'Í': 'I', 'Ì': 'I', 'Î': 'I',
#     'Ñ': 'N', 'Ç': 'C'
# })

# # Pre-define delimiter sets for faster lookups (Artist-specific)
# ARTIST_WORD_DELIMITERS = {
#     ' FT ', ' FT. ', ' FEAT ', ' FEAT. ', ' WITH ', ' AND ',
#     ' VS ', ' VS. ', ' FEATURING ', ' EN ',
#     ' AKA ', ' A.K.A. ', ' AKA. '
# }

# CHARACTER_DELIMITERS = {'|', '/', '#', '\\', ',', ';', '`', '_', '&', '+'}

# # Bracket processing setup
# BRACKET_PAIRS = {'(': ')', '[': ']', '{': '}'}
# OPENING_BRACKETS = set(BRACKET_PAIRS.keys())
# CLOSING_BRACKETS = set(BRACKET_PAIRS.values())

# def clean_apra_artist_name_optimized(name):
#     """Ultra-optimized version to clean APRA artist names using pre-compiled patterns and translation tables."""
#     if name is None or pd.isna(name):
#         return ""
        
#     try:
#         name = str(name).upper()
        
#         name_no_apostrophes = APOSTROPHE_PATTERN.sub('', name)
#         name_with_spaces = SPECIAL_CHARS_PATTERN.sub('', name_no_apostrophes)
#         name_with_spaces = HYPHEN_PATTERN.sub(' ', name_with_spaces)
#         name_with_spaces = QUOTE_PATTERN.sub('', name_with_spaces)
        
#         cleaned = NON_ALPHANUM_PATTERN.sub('', name_with_spaces)
#         cleaned = MULTIPLE_SPACES_PATTERN.sub(' ', cleaned).strip()
        
#         if not cleaned:
#             return name
            
#         for pattern in STOP_WORDS_PATTERNS:
#             cleaned = pattern.sub(' ', cleaned)
            
#         cleaned = MULTIPLE_SPACES_PATTERN.sub(' ', cleaned).strip()
        
#         if not cleaned:
#             return name_with_spaces.strip()
            
#         return cleaned
#     except:
#         return ""

# def normalize_special_chars_ultra_optimized(text):
#     """Ultra-optimized version to normalize special characters in APRA artist names using translation tables and pre-compiled patterns."""
#     if not text:
#         return ""
        
#     try:
#         text = str(text).upper()
        
#         text = LEADING_BRACKETS_PATTERN.sub('', text)
#         text = APOSTROPHE_HYPHEN_PATTERN.sub('', text)
        
#         text = text.translate(ACCENT_TRANSLATION)
            
#         return text
#     except:
#         return ""

# def balance_brackets_optimized(text):
#     """Optimized bracket balancing with early returns."""
#     if not text:
#         return ""
        
#     if not any(char in text for char in OPENING_BRACKETS | CLOSING_BRACKETS):
#         return text
        
#     stack = []
#     result = list(text)
        
#     i = 0
#     while i < len(result):
#         char = result[i]
        
#         if char in OPENING_BRACKETS:
#             stack.append((char, i))
#         elif char in CLOSING_BRACKETS:
#             if not stack:
#                 result[i] = ''
#                 i += 1
#                 continue
#             else:
#                 last_open, _ = stack[-1]
#                 expected_close = BRACKET_PAIRS[last_open]
                
#                 if char == expected_close:
#                     stack.pop()
#                 else:
#                     result[i] = expected_close
#                     stack.pop()
                
#         i += 1
        
#     while stack:
#         open_bracket, _ = stack.pop()
#         result.append(BRACKET_PAIRS[open_bracket])
        
#     return ''.join(result)

# def split_apra_artist_name_optimized(name):
#     """Optimized version to split APRA artist names with reduced string operations and faster delimiter checking."""
#     if not name:
#         return []
        
#     try:
#         original_name = str(name).upper() if name is not None else ""
#         name_normalized = original_name.translate(str.maketrans("'?", "  ")).strip() # Using translate for faster removal
#         name_normalized = balance_brackets_optimized(name_normalized)
#     except:
#         return []
        
#     all_parts = []
#     bracket_contents = []
#     working_name = name_normalized
        
#     for open_bracket, close_bracket in BRACKET_PAIRS.items():
#         start_pos = working_name.find(open_bracket)
#         while start_pos != -1:
#             end_pos = working_name.find(close_bracket, start_pos + 1)
#             if end_pos != -1:
#                 bracket_content = working_name[start_pos + 1:end_pos].strip()
#                 if bracket_content:
#                     bracket_contents.append(bracket_content)
#                 working_name = working_name[:start_pos] + working_name[end_pos + 1:]
#             else:
#                 start_pos = working_name.find(open_bracket, start_pos + 1)
#                 continue
#             start_pos = working_name.find(open_bracket)
        
#     main_part = working_name.strip()
#     main_parts = []
#     has_explicit_delimiters = False
        
#     if main_part:
#         main_with_boundaries = f' {main_part} ' # Add spaces for easier word delimiter matching
        
#         if any(d in main_with_boundaries for d in CHARACTER_DELIMITERS):
#             has_explicit_delimiters = True
#         elif any(wd in main_with_boundaries for wd in ARTIST_WORD_DELIMITERS):
#             has_explicit_delimiters = True
        
#         delimiter_chars = ''.join(CHARACTER_DELIMITERS)
#         delimiter_translation = str.maketrans(delimiter_chars, '|' * len(delimiter_chars))
#         main_with_boundaries = main_with_boundaries.translate(delimiter_translation)
        
#         for wd in ARTIST_WORD_DELIMITERS:
#             if wd in main_with_boundaries:
#                 main_with_boundaries = main_with_boundaries.replace(wd, '|')
        
#         main_parts = [part.strip() for part in main_with_boundaries.split('|') if part.strip()]
#         for part in main_parts:
#             clean_part = normalize_special_chars_ultra_optimized(part)
#             if clean_part and len(clean_part) > 1:
#                 all_parts.append(clean_part)
        
#     for content in bracket_contents:
#         delimiter_word_at_start = next((dw for dw in ARTIST_WORD_DELIMITERS
#                                         if content.startswith(dw.strip() + ' ')), None)
        
#         if delimiter_word_at_start:
#             rest_of_content = content[len(delimiter_word_at_start):].strip()
            
#             has_more_delimiters = (any(d in rest_of_content for d in CHARACTER_DELIMITERS) or
#                                    any(wd in f" {rest_of_content} " for wd in ARTIST_WORD_DELIMITERS))
            
#             if has_more_delimiters:
#                 delimiter_chars = '|/,&+-' # Explicit common chars
#                 cleaned = rest_of_content.translate(str.maketrans(delimiter_chars, '|' * len(delimiter_chars)))
#                 for wd in ARTIST_WORD_DELIMITERS:
#                     if wd in f" {cleaned} ": # Check with boundary spaces
#                         cleaned = cleaned.replace(wd.strip(), '|')
                
#                 for part in cleaned.split('|'):
#                     part = part.strip()
#                     if part:
#                         clean_part = normalize_special_chars_ultra_optimized(part)
#                         if clean_part and len(clean_part) > 1:
#                             all_parts.append(clean_part)
#             else:
#                 clean_part = normalize_special_chars_ultra_optimized(rest_of_content)
#                 if clean_part and len(clean_part) > 1:
#                     all_parts.append(clean_part)
#         else:
#             has_delimiter = (any(d in content for d in CHARACTER_DELIMITERS) or
#                              any(wd in f" {content} " for wd in ARTIST_WORD_DELIMITERS))
            
#             if has_delimiter:
#                 delimiter_chars = '|/,&+-' # Explicit common chars
#                 cleaned = content.translate(str.maketrans(delimiter_chars, '|' * len(delimiter_chars)))
#                 for wd in ARTIST_WORD_DELIMITERS:
#                     if wd in f" {cleaned} ": # Check with boundary spaces
#                         cleaned = cleaned.replace(wd.strip(), '|')
                
#                 for part in cleaned.split('|'):
#                     part = part.strip()
#                     if part:
#                         clean_part = normalize_special_chars_ultra_optimized(part)
#                         if clean_part and len(clean_part) > 1:
#                             all_parts.append(clean_part)
#             else:
#                 clean_part = normalize_special_chars_ultra_optimized(content)
#                 if clean_part and len(clean_part) > 1:
#                     all_parts.append(clean_part)
        
#     if len(main_parts) == 2 and not has_explicit_delimiters:
#         reversed_name = f"{main_parts[1]} {main_parts[0]}"
#         clean_reversed = normalize_special_chars_ultra_optimized(reversed_name)
#         if clean_reversed and clean_reversed != normalize_special_chars_ultra_optimized(main_part):
#             all_parts.append(clean_reversed)
        
#     if len(all_parts) == 0 and main_part:
#         clean_full_name = normalize_special_chars_ultra_optimized(main_part)
#         if clean_full_name and len(clean_full_name) > 1:
#             all_parts.append(clean_full_name)
        
#     if len(main_parts) == 1 and main_part:
#         clean_full_name = normalize_special_chars_ultra_optimized(main_part)
#         if clean_full_name and len(clean_full_name) > 1:
#             if clean_full_name not in all_parts:
#                 all_parts.append(clean_full_name)
        
#     unique_parts = []
#     seen = set()
#     for part in all_parts:
#         clean_part = part.replace('(', '').replace(')', '').replace('?', '')
#         if clean_part and clean_part not in seen and len(clean_part) > 1:
#             seen.add(clean_part)
#             unique_parts.append(clean_part)
        
#     return unique_parts

# def preprocess_apra_artist_dataframe_optimized(df, name_col='APRA_ARTIST_NAME', id_col='APRA_ARTIST_ID'):
#     """Optimized preprocessing of APRA artist data - includes RDC_WORKS_ID."""
#     df_pandas = df.to_pandas()
#     total_rows = len(df_pandas)
        
#     print(f"Processing {total_rows} rows in optimized mode...")
        
#     df_pandas[name_col] = df_pandas[name_col].fillna("")
        
#     print("Cleaning artist names...")
#     df_pandas['CLEAN_NAME'] = df_pandas[name_col].apply(clean_apra_artist_name_optimized)
        
#     print("Normalizing names...")
#     df_pandas['NORMALIZED_NAME'] = df_pandas[name_col].apply(normalize_special_chars_ultra_optimized)
        
#     print("Splitting artist names...")
#     df_pandas['DELIMITED_PARTS'] = df_pandas[name_col].apply(split_apra_artist_name_optimized)
        
#     result_df = df.session.create_dataframe(df_pandas)
        
#     return result_df

# def create_expanded_apra_artist_dataframe_optimized(df, name_col='APRA_ARTIST_NAME', id_col='APRA_ARTIST_ID'):
#     """Optimized expansion of APRA artist dataframe with artist variants - includes RDC_WORKS_ID."""
#     df_pandas = df.to_pandas()
#     total_rows = len(df_pandas)
        
#     print(f"Expanding {total_rows} rows in optimized mode...")
        
#     expanded_rows = []
        
#     original_names_upper = np.array([
#         str(name).upper().replace("'", "") if name is not None else ""
#         for name in df_pandas[name_col].values
#     ])
        
#     for idx, (_, row) in enumerate(df_pandas.iterrows()):
#         original_row = row.to_dict()
#         original_row['IS_VARIANT'] = 'ORIGINAL'
#         expanded_rows.append(original_row)
            
#         if 'DELIMITED_PARTS' in row and row['DELIMITED_PARTS']:
#             original_name_upper = original_names_upper[idx]
                
#             for part in row['DELIMITED_PARTS']:
#                 if part and part != original_name_upper and len(part) > 2:
#                     variant_row = row.to_dict()
#                     variant_row[name_col] = part
#                     variant_row['CLEAN_NAME'] = clean_apra_artist_name_optimized(part)
#                     variant_row['NORMALIZED_NAME'] = normalize_special_chars_ultra_optimized(part)
#                     variant_row['IS_VARIANT'] = 'VARIANT'
#                     variant_row['DELIMITED_PARTS'] = []
#                     # Retain RDC_WORKS_ID and MUZOOKA_TRACK_ID
#                     if 'RDC_WORKS_ID' in row:
#                         variant_row['RDC_WORKS_ID'] = row['RDC_WORKS_ID']
#                     if 'MUZOOKA_TRACK_ID' in row:
#                         variant_row['MUZOOKA_TRACK_ID'] = row['MUZOOKA_TRACK_ID']
#                     expanded_rows.append(variant_row)
        
#     expanded_df = pd.DataFrame(expanded_rows)
#     result_df = df.session.create_dataframe(expanded_df)
        
#     return result_df

# def filter_apra_artist_by_work_ids(session, table_name, work_ids):
#     """Filter APRA artist data by work IDs."""
#     work_ids_str = ", ".join([f"'{work_id}'" for work_id in work_ids])
        
#     query = f"""
#     SELECT *
#     FROM {table_name}
#     WHERE APRA_WORK_ID IN ({work_ids_str})
#     """
        
#     return session.sql(query)

# def process_apra_artists_optimized(session, mode="full", work_ids=None, sample_percentage=1):
#     """Optimized APRA artist processing - includes RDC_WORKS_ID."""
        
#     print(f"=== OPTIMIZED APRA ARTIST PROCESSING ===")
#     print(f"Running in {mode} mode with optimized processing")
        
#     # Column names for the APRA Artist table
#     name_col_name = 'APRA_ARTIST_NAME'
#     id_col_name = 'APRA_ARTIST_ID' # Or 'RDC_WORKS_ID' if that's the primary key for your join logic

#     if mode == "example":
#         print("Creating example APRA artist data...")
#         apra_artist_examples = pd.DataFrame({
#             'RDC_WORKS_ID': [101, 102, 103, 104, 105, 106, 107],
#             'APRA_WORK_ID': ['APRA001', 'APRA002', 'APRA003', 'APRA004', 'APRA005', 'APRA006', 'APRA007'],
#             'APRA_ARTIST_ID': ['ART001', 'ART002', 'ART003', 'ART004', 'ART005', 'ART006', 'ART007'],
#             'MUZOOKA_TRACK_ID': ['TRK001', 'TRK002', 'TRK003', 'TRK004', 'TRK005', 'TRK006', 'TRK007'],
#             'APRA_ARTIST_NAME': [
#                 'QUEEN',
#                 'DAFT PUNK (FEAT. PHARRELL WILLIAMS)',
#                 'LED ZEPPELIN / ATLANTIC RECORDS',
#                 'ARIANA GRANDE FT. THE WEEKND',
#                 'BTS (BANGTAN BOYS)',
#                 'TAYLOR SWIFT AKA ANONYMOUS',
#                 'DRAKE & RIHANNA'
#             ]
#         })
        
#         apra_artist_df = session.create_dataframe(apra_artist_examples)
        
#         print("\nPreprocessing APRA artist data (OPTIMIZED)...")
#         apra_artist_preprocessed = preprocess_apra_artist_dataframe_optimized(apra_artist_df, name_col=name_col_name, id_col=id_col_name)
        
#         print("\nCreating expanded APRA artist dataframe (OPTIMIZED)...")
#         apra_artist_expanded = create_expanded_apra_artist_dataframe_optimized(apra_artist_preprocessed, name_col=name_col_name, id_col=id_col_name)
        
#         # Display results
#         print("\nPreprocessed APRA Artist Data:")
#         apra_artist_preprocessed_pandas = apra_artist_preprocessed.to_pandas()
#         for _, row in apra_artist_preprocessed_pandas.iterrows():
#             print(f"RDC_WORKS_ID: {row.get('RDC_WORKS_ID', 'N/A')}, APRA_ARTIST_ID: {row[id_col_name]}, Name: {row[name_col_name]}")
#             print(f"  Clean Name: {row['CLEAN_NAME']}")
#             print(f"  Normalized Name: {row['NORMALIZED_NAME']}")
#             print(f"  Delimited Parts: {row['DELIMITED_PARTS']}")
#             print()
            
#         print("\nExpanded APRA Artist Data:")
#         apra_artist_expanded_pandas = apra_artist_expanded.to_pandas()
#         for _, row in apra_artist_expanded_pandas.iterrows():
#             print(f"RDC_WORKS_ID: {row.get('RDC_WORKS_ID', 'N/A')}, APRA_ARTIST_ID: {row[id_col_name]}, Name: {row[name_col_name]}, Variant: {row['IS_VARIANT']}")
#             print(f"  Clean Name: {row['CLEAN_NAME']}")
#             print()
        
#         return apra_artist_expanded
        
#     elif mode == "subset":
#         print(f"Running with {sample_percentage}% of APRA Artist data using SQL SAMPLE (OPTIMIZED)")
        
#         apra_artist_table = "EDW_APPS.MATCHING.ADC_WORKS_TRACKS_MATCHED_ARTISTS"
#         apra_artist_preprocessed_table = "EDW_APPS.MATCHING.ADC_APRA_ARTISTS_PREPROCESSED_TEMP_TEST" # New temp table name
#         apra_artist_expanded_table = "EDW_APPS.MATCHING.ADC_APRA_ARTISTS_EXPANDED_CLEAN" # New output table name
        
#         try:
#             # Select all columns to ensure RDC_WORKS_ID and MUZOOKA_TRACK_ID are included
#             apra_artist_query = f"""
#             SELECT *
#             FROM {apra_artist_table}
#             SAMPLE ({sample_percentage})
#             """
            
#             print(f"Executing query: {apra_artist_query}")
#             apra_artist_df = session.sql(apra_artist_query)
#             apra_artist_count = apra_artist_df.count()
#             print(f"APRA Artist sample contains {apra_artist_count} rows")
            
#             if apra_artist_count == 0:
#                 print("Warning: APRA Artist DataFrame is empty.")
#                 return None
            
#             print("\nPreprocessing APRA artist data (OPTIMIZED)...")
#             apra_artist_preprocessed = preprocess_apra_artist_dataframe_optimized(apra_artist_df, name_col=name_col_name, id_col=id_col_name)
            
#             print(f"Saving preprocessed APRA artist data to {apra_artist_preprocessed_table}...")
#             apra_artist_preprocessed.write.mode("overwrite").save_as_table(apra_artist_preprocessed_table)
            
#             print("\nCreating expanded APRA artist dataframe (OPTIMIZED)...")
#             apra_artist_preprocessed = session.table(apra_artist_preprocessed_table) # Reload to ensure all columns are correct
#             apra_artist_expanded = create_expanded_apra_artist_dataframe_optimized(apra_artist_preprocessed, name_col=name_col_name, id_col=id_col_name)
            
#             print(f"Saving expanded APRA artist data to {apra_artist_expanded_table}...")
#             apra_artist_expanded.write.mode("overwrite").save_as_table(apra_artist_expanded_table)
            
#             print("\nCleaning up intermediate tables...")
#             session.sql(f"DROP TABLE IF EXISTS {apra_artist_preprocessed_table}").collect()
            
#             print("Optimized APRA artist preprocessing completed successfully!")
#             print(f"Final results: {apra_artist_count} APRA artists processed")
#             print(f"Output table: {apra_artist_expanded_table}")
            
#             # Show sample of results with RDC_WORKS_ID
#             print("\nSample of processed data:")
#             sample_query = f"""
#             SELECT
#                 APRA_ARTIST_ID,
#                 APRA_WORK_ID,
#                 RDC_WORKS_ID,
#                 MUZOOKA_TRACK_ID,
#                 APRA_ARTIST_NAME,
#                 CLEAN_NAME,
#                 NORMALIZED_NAME,
#                 IS_VARIANT
#             FROM {apra_artist_expanded_table}
#             LIMIT 10
#             """
#             session.sql(sample_query).show()
            
#             return apra_artist_expanded
            
#         except Exception as e:
#             print(f"Error during APRA artist processing: {str(e)}")
#             import traceback
#             traceback.print_exc()
#             return None

#     elif mode == "full":
#         print("Running with FULL APRA Artist data using SQL SAMPLE (OPTIMIZED)")
        
#         apra_artist_table = "EDW_APPS.MATCHING.ADC_WORKS_TRACKS_MATCHED_ARTISTS"
#         apra_artist_preprocessed_table = "EDW_APPS.MATCHING.ADC_APRA_ARTISTS_PREPROCESSED_TEMP" # New temp table name
#         apra_artist_expanded_table = "EDW_APPS.MATCHING.ADC_APRA_ARTISTS_CLEAN" # New output table name
        
#         try:
#             # Select all columns
#             apra_artist_query = f"""
#             SELECT *
#             FROM {apra_artist_table}
#             """
            
#             print(f"Executing query: {apra_artist_query}")
#             apra_artist_df = session.sql(apra_artist_query)
#             apra_artist_count = apra_artist_df.count()
#             print(f"APRA Artist FULL dataset contains {apra_artist_count} rows")
            
#             if apra_artist_count == 0:
#                 print("Warning: APRA Artist DataFrame is empty.")
#                 return None
            
#             print("\nPreprocessing APRA artist data (OPTIMIZED)...")
#             apra_artist_preprocessed = preprocess_apra_artist_dataframe_optimized(apra_artist_df, name_col=name_col_name, id_col=id_col_name)
            
#             print(f"Saving preprocessed APRA artist data to {apra_artist_preprocessed_table}...")
#             apra_artist_preprocessed.write.mode("overwrite").save_as_table(apra_artist_preprocessed_table)
            
#             print("\nCreating expanded APRA artist dataframe (OPTIMIZED)...")
#             apra_artist_preprocessed = session.table(apra_artist_preprocessed_table) # Reload to ensure all columns are correct
#             apra_artist_expanded = create_expanded_apra_artist_dataframe_optimized(apra_artist_preprocessed, name_col=name_col_name, id_col=id_col_name)
            
#             print(f"Saving expanded APRA artist data to {apra_artist_expanded_table}...")
#             apra_artist_expanded.write.mode("overwrite").save_as_table(apra_artist_expanded_table)
            
#             print("\nCleaning up intermediate tables...")
#             session.sql(f"DROP TABLE IF EXISTS {apra_artist_preprocessed_table}").collect()
            
#             print("Optimized APRA artist preprocessing completed successfully!")
#             print(f"Final results: {apra_artist_count} APRA artists processed")
#             print(f"Output table: {apra_artist_expanded_table}")
            
#             return apra_artist_expanded
            
#         except Exception as e:
#             print(f"Error during APRA artist processing: {str(e)}")
#             import traceback
#             traceback.print_exc()
#             return None


# if __name__ == "__main__":
#      result = process_apra_artists_optimized(session, mode="subset", sample_percentage=10)

In [ ]:
# # ENHANCED DELIMITER COUNTS SCRIPT WITH TOTAL ARTIST LOGIC



# import pandas as pd
# import numpy as np
# from snowflake.snowpark import Session
# from snowflake.snowpark.functions import col, array_size, split, length, when, count, window
# from snowflake.snowpark import Window

# def get_delimited_parts_count(session, table_name, delimited_parts_col):
    
#     # Get a sample row to check the format
#     sample_query = f"SELECT {delimited_parts_col} FROM {table_name} WHERE {delimited_parts_col} IS NOT NULL LIMIT 1"
#     sample_result = session.sql(sample_query).collect()
    
#     if not sample_result:
#         return f"LENGTH({delimited_parts_col})"  # Fallback
    
#     sample_row = sample_result[0][delimited_parts_col]
    
#     # Determine if it's an array or string
#     if isinstance(sample_row, list):
#         # It's an actual array
#         return f"ARRAY_SIZE({delimited_parts_col})"
#     elif isinstance(sample_row, str) and (sample_row.startswith('[') and sample_row.endswith(']')):
#         # It's a string representation of an array, can use PARSE_JSON
#         return f"ARRAY_SIZE(PARSE_JSON({delimited_parts_col}))"
#     elif isinstance(sample_row, str) and (',' in sample_row):
#         # It's a string with comma delimiters
#         return f"ARRAY_SIZE(SPLIT({delimited_parts_col}, ','))"
#     else:
#         # For any other case, use a simple word count as an approximation
#         return f"ARRAY_SIZE(SPLIT(TRIM({delimited_parts_col}), ' '))"

# def add_delimiter_count_adc_artists(session):
#     """
#     Add delimiter count and UNIQUE total artist count to ADC/APRA artists dataset.
#     """
#     # Define table names
#     primary_artist_table = "EDW_APPS.MATCHING.ADC_APRA_ARTISTS_EXPANDED_CLEAN"
#     output_table = "EDW_APPS.MATCHING.ADC_ARTIST_DELIMITER_COUNT_CLEAN"
    
#     # Determine how to count delimited parts
#     primary_delimited_parts_col = 'DELIMITED_PARTS'
#     primary_delimited_parts_count = get_delimited_parts_count(
#         session, primary_artist_table, primary_delimited_parts_col
#     )
    
#     # Execute query to add delimiter count and UNIQUE total artist count to ADC artists
#     delimiter_count_query = f"""
#     WITH artist_counts AS (
#         SELECT 
#             *,
#             {primary_delimited_parts_count} AS APRA_ARTIST_CT
#         FROM {primary_artist_table}
#     ),
#     flattened_artists AS (
#         SELECT 
#             TRIM(artist_name.value::STRING) AS INDIVIDUAL_ARTIST_NAME,
#             ac.*
#         FROM artist_counts ac,
#         LATERAL FLATTEN(
#             input => CASE 
#                 WHEN DELIMITED_PARTS LIKE '[%]' THEN PARSE_JSON(DELIMITED_PARTS)
#                 WHEN DELIMITED_PARTS LIKE '%,%' THEN SPLIT(DELIMITED_PARTS, ',')
#                 ELSE ARRAY_CONSTRUCT(DELIMITED_PARTS)
#             END,
#             outer => true
#         ) AS artist_name
#         WHERE TRIM(artist_name.value::STRING) IS NOT NULL 
#         AND TRIM(artist_name.value::STRING) != ''
#     ),
#     unique_artists_per_work AS (
#         SELECT 
#             APRA_WORK_ID,
#             COUNT(DISTINCT UPPER(TRIM(INDIVIDUAL_ARTIST_NAME))) AS APRA_TOTAL_ARTISTS
#         FROM flattened_artists
#         GROUP BY APRA_WORK_ID
#     )
#     SELECT 
#         ac.*,
#         COALESCE(uapw.APRA_TOTAL_ARTISTS, 0) AS APRA_TOTAL_ARTISTS
#     FROM artist_counts ac
#     LEFT JOIN unique_artists_per_work uapw ON ac.APRA_WORK_ID = uapw.APRA_WORK_ID
#     """
    
#     # Execute the query and save results
#     results = session.sql(delimiter_count_query)
#     results.write.mode("overwrite").save_as_table(output_table)
    
#     print(f"ADC artists with delimiter counts and UNIQUE total artist counts saved to {output_table}")

# def add_delimiter_count_mzk_artists(session):
#     """
#     Add delimiter count and UNIQUE total artist count to MZK artists dataset.
#     """
#     secondary_artist_table = "EDW_APPS.MATCHING.MZK_TRACKS_ARTISTS_EXPANDED_CLEAN"
#     output_table = "EDW_APPS.MATCHING.MZK_ARTIST_DELIMITER_COUNT_CLEAN"  
    
#     # Determine how to count delimited parts for secondary artists
#     sec_delimited_parts_col = 'DELIMITED_PARTS'
#     sec_delimited_parts_count = get_delimited_parts_count(
#         session, secondary_artist_table, sec_delimited_parts_col
#     )
    
#     # Execute query to add delimiter count and UNIQUE total artist count to MZK artists
#     delimiter_count_query = f"""
#     WITH artist_counts AS (
#         SELECT 
#             *,
#             {sec_delimited_parts_count} AS MZK_ARTIST_CT
#         FROM {secondary_artist_table}
#     ),
#     flattened_artists AS (
#         SELECT 
#             TRIM(artist_name.value::STRING) AS INDIVIDUAL_ARTIST_NAME,
#             ac.*
#         FROM artist_counts ac,
#         LATERAL FLATTEN(
#             input => CASE 
#                 WHEN DELIMITED_PARTS LIKE '[%]' THEN PARSE_JSON(DELIMITED_PARTS)
#                 WHEN DELIMITED_PARTS LIKE '%,%' THEN SPLIT(DELIMITED_PARTS, ',')
#                 ELSE ARRAY_CONSTRUCT(DELIMITED_PARTS)
#             END,
#             outer => true
#         ) AS artist_name
#         WHERE TRIM(artist_name.value::STRING) IS NOT NULL 
#         AND TRIM(artist_name.value::STRING) != ''
#     ),
#     unique_artists_per_track AS (
#         SELECT 
#             MUZOOKA_TRACK_ID,
#             COUNT(DISTINCT UPPER(TRIM(INDIVIDUAL_ARTIST_NAME))) AS MZK_TOTAL_ARTISTS
#         FROM flattened_artists
#         GROUP BY MUZOOKA_TRACK_ID
#     )
#     SELECT 
#         ac.*,
#         COALESCE(uapt.MZK_TOTAL_ARTISTS, 0) AS MZK_TOTAL_ARTISTS
#     FROM artist_counts ac
#     LEFT JOIN unique_artists_per_track uapt ON ac.MUZOOKA_TRACK_ID = uapt.MUZOOKA_TRACK_ID
#     """
    
#     # Execute the query and save results
#     results = session.sql(delimiter_count_query)
#     results.write.mode("overwrite").save_as_table(output_table)
    
#     print(f"MZK artists with delimiter counts and UNIQUE total artist counts saved to {output_table}")

# def main(session):
   
#     print("Starting enhanced delimiter counts processing with total artist logic...")
    
#     # Add delimiter counts and total artist counts to both datasets
#     print("\n1. Processing ADC/APRA artists...")
#     add_delimiter_count_adc_artists(session)
    
#     print("\n2. Processing MZK artists...")
#     add_delimiter_count_mzk_artists(session)
    
#     print("\nEnhanced delimiter counts processing complete!")
#     print("Results include:")
#     print("- APRA_ARTIST_CT: Individual artist delimiter count")  
#     print("- APRA_TOTAL_ARTISTS: UNIQUE count of all artists per APRA work (duplicates removed)")
#     print("- MZK_ARTIST_CT: Individual artist delimiter count") 
#     print("- MZK_TOTAL_ARTISTS: UNIQUE count of all artists per Muzooka track (duplicates removed)")

# if __name__ == "__main__":
#     main(session)

In [ ]:
# #normal matching 


# def create_enhanced_udf(session):
#     """Create enhanced JavaScript UDF for artist name matching with multiple format handling"""
    
#     create_udf_sql = """
#     CREATE OR REPLACE FUNCTION ENHANCED_MATCH_ARTIST_NAMES(name1 STRING, name2 STRING)
#     RETURNS OBJECT
#     LANGUAGE JAVASCRIPT
#     AS
#     $$
    
#     /**
#      * Normalize names to handle special cases:
#      * 1. Remove/normalize generational suffixes
#      * 2. Replace hyphens with spaces to handle hyphenated names
#      * 3. Normalize special prefixes
#      * 4. Remove bracket content
#      */
#     function normalizeNameForComparison(name) {
#         if (!name) return "";
        
#         // Remove content in brackets first
#         name = name.replace(/\\([^\\)]*\\)/g, "");
        
#         name = name.toUpperCase().trim();
        
#         // 1. Remove generational suffixes: I, II, III, IV, V and their variations
#         name = name.replace(/\\b(I{1,3}|IV|V|1ST|2ND|3RD|[4-9]TH)\\b$/g, "").trim();
        
#         // 2. Replace hyphens with spaces for better matching
#         name = name.replace(/-/g, " ");
        
#         // 3. Special handling for prefixes - ensure consistent spacing
#         const special_prefixes = ["VAN", "VON", "DE", "DER", "LA", "LE", "DI", "DEL", "DOS", "DA", "DU", "AL", "EL", "JESUS", "CHAVEZ"];
        
#         // Create regex patterns for each prefix to match only when it's a standalone word
#         for (const prefix of special_prefixes) {
#             const regex = new RegExp(`\\\\b${prefix}\\\\b`, 'g');
#             // Ensure consistent spacing for each prefix
#             name = name.replace(regex, prefix);
#         }
        
#         // Remove extra whitespace
#         name = name.replace(/\\s+/g, " ").trim();
        
#         return name;
#     }

#     /**
#      * Enhanced name parsing that handles multiple formats
#      */
#     function parseNameComponents(name) {
#         if (!name || name === "") {
#             return {formats: []};
#         }

#         // Normalize the name first
#         name = normalizeNameForComparison(name);

#         // Special suffixes and prefixes
#         const suffixes = ["SENIOR", "SNR", "SR", "JUNIOR", "JNR", "JR"];
#         const special_prefixes = ["VAN", "VON", "DE", "DER", "LA", "LE", "DI", "DEL", "DOS", "DA", "DU", "AL", "EL", "JESUS", "CHAVEZ"];

#         // Clean and split the name
#         let parts = name.split(/\\s+/);

#         // Handle name with no spaces
#         if (parts.length === 1) {
#             return {formats: [{first: "", last: parts[0], middle: "", suffix: "", is_initial: false}]};
#         }

#         // Check for and extract suffixes
#         let suffix = "";
#         for (const suffixTerm of suffixes) {
#             const suffixIndex = parts.findIndex(part => part === suffixTerm);
#             if (suffixIndex !== -1) {
#                 suffix = parts[suffixIndex];
#                 parts.splice(suffixIndex, 1);
#                 break;
#             }
#         }

#         // After removing suffix, check if we're back to a single name
#         if (parts.length === 1) {
#             return {formats: [{first: "", last: parts[0], middle: "", suffix: suffix, is_initial: false}]};
#         }

#         // Generate multiple format possibilities
#         const formatsToCheck = [];
        
#         // Check for special multi-word last names with prefixes like "VAN BEETHOVEN"
#         let hasSpecialPrefix = false;
#         let specialPrefixIndex = -1;
        
#         for (let i = 0; i < parts.length; i++) {
#             if (special_prefixes.includes(parts[i])) {
#                 hasSpecialPrefix = true;
#                 specialPrefixIndex = i;
#                 break;
#             }
#         }
        
#         if (hasSpecialPrefix) {
#             if (specialPrefixIndex === 0 && parts.length >= 2) {
#                 // Case: "VAN BEETHOVEN" or "VAN BEETHOVEN LUDWIG"
#                 if (parts.length === 2) {
#                     // Just "VAN BEETHOVEN" - treat as compound last name
#                     formatsToCheck.push({
#                         first: "",
#                         last: parts.join(" "),
#                         middle: "",
#                         suffix: suffix,
#                         is_initial: false
#                     });
#                 } else {
#                     // "BEETHOVEN LUDWIG VAN" - LASTNAME FIRSTNAME PREFIX format
#                     formatsToCheck.push({
#                         first: parts[1],
#                         last: parts[0] + " " + parts[2],
#                         middle: parts.slice(3).join(" "),
#                         suffix: suffix,
#                         is_initial: parts[1] && parts[1].length === 1
#                     });
                    
#                     // "VAN BEETHOVEN LUDWIG" - PREFIX LASTNAME FIRSTNAME format
#                     const lastName = parts.slice(0, 2).join(" ");
#                     const firstName = parts[2];
#                     const middleName = parts.slice(3).join(" ");
                    
#                     formatsToCheck.push({
#                         first: firstName,
#                         last: lastName,
#                         middle: middleName,
#                         suffix: suffix,
#                         is_initial: firstName && firstName.length === 1
#                     });
#                 }
#             } else if (specialPrefixIndex > 0) {
#                 // Case: "LUDWIG VAN BEETHOVEN" - FIRSTNAME PREFIX LASTNAME format
#                 const firstName = parts.slice(0, specialPrefixIndex).join(" ");
#                 const lastName = parts.slice(specialPrefixIndex).join(" ");
                
#                 formatsToCheck.push({
#                     first: firstName,
#                     last: lastName,
#                     middle: "",
#                     suffix: suffix,
#                     is_initial: firstName.length === 1
#                 });
                
#                 // Also try reversed: LASTNAME FIRSTNAME format (VAN BEETHOVEN LUDWIG)
#                 formatsToCheck.push({
#                     first: firstName,
#                     last: lastName,
#                     middle: "",
#                     suffix: suffix,
#                     is_initial: firstName.length === 1
#                 });
#             }
#         }
        
#         // Standard format handling for all cases
#         if (parts.length === 2) {
#             // Two parts: ALWAYS try both orders for cases like "BERNIE BEN" vs "BEN BERNIE"
#             formatsToCheck.push({
#                 first: parts[0],
#                 last: parts[1],
#                 middle: "",
#                 suffix: suffix,
#                 is_initial: parts[0].length === 1
#             });
            
#             formatsToCheck.push({
#                 first: parts[1],
#                 last: parts[0],
#                 middle: "",
#                 suffix: suffix,
#                 is_initial: parts[1].length === 1
#             });
#         } else if (parts.length >= 3) {
#             // Format 1: LASTNAME FIRSTNAME MIDDLENAME (ADC Style)
#             formatsToCheck.push({
#                 first: parts[1],
#                 last: parts[0],
#                 middle: parts.slice(2).join(" "),
#                 suffix: suffix,
#                 is_initial: parts[1].length === 1
#             });
            
#             // Format 2: FIRSTNAME MIDDLENAME LASTNAME (Muzooka Style)
#             formatsToCheck.push({
#                 first: parts[0],
#                 last: parts[parts.length - 1],
#                 middle: parts.slice(1, parts.length - 1).join(" "),
#                 suffix: suffix,
#                 is_initial: parts[0].length === 1
#             });
            
#             // Format 3: FIRSTNAME LASTNAME (treating middle as part of first)
#             formatsToCheck.push({
#                 first: parts.slice(0, parts.length - 1).join(" "),
#                 last: parts[parts.length - 1],
#                 middle: "",
#                 suffix: suffix,
#                 is_initial: false
#             });
            
#             // Format 4: LASTNAME FIRSTNAME (treating middle as part of last)
#             formatsToCheck.push({
#                 first: parts[parts.length - 1],
#                 last: parts.slice(0, parts.length - 1).join(" "),
#                 middle: "",
#                 suffix: suffix,
#                 is_initial: parts[parts.length - 1].length === 1
#             });
            
#             // Format 5: For compound last names (first two parts as last name)
#             if (parts.length >= 3) {
#                 formatsToCheck.push({
#                     first: parts.slice(2).join(" "),
#                     last: parts.slice(0, 2).join(" "),
#                     middle: "",
#                     suffix: suffix,
#                     is_initial: false
#                 });
#             }
            
#             // Format 6: For compound last names (last two parts as last name)
#             if (parts.length >= 3) {
#                 formatsToCheck.push({
#                     first: parts.slice(0, parts.length - 2).join(" "),
#                     last: parts.slice(parts.length - 2).join(" "),
#                     middle: "",
#                     suffix: suffix,
#                     is_initial: false
#                 });
#             }
#         }

#         return {formats: formatsToCheck};
#     }

#     /**
#      * Calculate string similarity using Levenshtein distance
#      */
#     function calculateStringSimilarity(s1, s2) {
#         if (!s1 || !s2) return 0;
#         if (s1 === s2) return 1.0;

#         const str1 = s1.toUpperCase().trim();
#         const str2 = s2.toUpperCase().trim();

#         if (str1 === str2) return 1.0;
#         if (str1 === "" || str2 === "") return 0.0;

#         const len1 = str1.length;
#         const len2 = str2.length;
#         const matrix = Array(len1 + 1).fill().map(() => Array(len2 + 1).fill(0));

#         for (let i = 0; i <= len1; i++) matrix[i][0] = i;
#         for (let j = 0; j <= len2; j++) matrix[0][j] = j;

#         for (let i = 1; i <= len1; i++) {
#             for (let j = 1; j <= len2; j++) {
#                 const cost = str1[i-1] === str2[j-1] ? 0 : 1;
#                 matrix[i][j] = Math.min(
#                     matrix[i-1][j] + 1,
#                     matrix[i][j-1] + 1,
#                     matrix[i-1][j-1] + cost
#                 );
#             }
#         }

#         const distance = matrix[len1][len2];
#         const maxLen = Math.max(len1, len2);
#         return 1 - (distance / maxLen);
#     }

#     /**
#      * Jaro-Winkler similarity implementation
#      */
#     function jaroWinklerSimilarity(s1, s2) {
#         if (!s1 || !s2 || s1.length === 0 || s2.length === 0) return 0;
#         if (s1 === s2) return 1;
        
#         let m = 0;
#         let t = 0;
#         let range = Math.floor(Math.max(s1.length, s2.length) / 2) - 1;
#         range = Math.max(0, range);
        
#         let s1Matches = new Array(s1.length).fill(false);
#         let s2Matches = new Array(s2.length).fill(false);
        
#         for (let i = 0; i < s1.length; i++) {
#             let start = Math.max(0, i - range);
#             let end = Math.min(i + range + 1, s2.length);
            
#             for (let j = start; j < end; j++) {
#                 if (!s2Matches[j] && s1[i] === s2[j]) {
#                     s1Matches[i] = true;
#                     s2Matches[j] = true;
#                     m++;
#                     break;
#                 }
#             }
#         }
        
#         if (m === 0) return 0;
        
#         let k = 0;
#         for (let i = 0; i < s1.length; i++) {
#             if (s1Matches[i]) {
#                 while (k < s2.length && !s2Matches[k]) k++;
#                 if (k < s2.length && s1[i] !== s2[k]) t++;
#                 if (k < s2.length) k++;
#             }
#         }
        
#         t = Math.floor(t / 2);
#         let jaroSim = (m / s1.length + m / s2.length + (m - t) / m) / 3;
        
#         let p = 0.1;
#         let l = 0;
        
#         for (let i = 0; i < Math.min(4, Math.min(s1.length, s2.length)); i++) {
#             if (s1[i] === s2[i]) {
#                 l++;
#             } else {
#                 break;
#             }
#         }
        
#         return jaroSim + (l * p * (1 - jaroSim));
#     }

#     /**
#      * Calculate match score between two name formats with enhanced logic
#      */
#     function calculateMatchScore(format1, format2) {
#         if (!format1 || !format2) return 0;

#         const first1 = format1.first || "";
#         const last1 = format1.last || "";
#         const middle1 = format1.middle || "";
        
#         const first2 = format2.first || "";
#         const last2 = format2.last || "";
#         const middle2 = format2.middle || "";

#         // Last name similarity (most important)
#         const lastNameScore = calculateStringSimilarity(last1, last2);
        
#         // First name similarity with enhanced initial handling
#         let firstNameScore = 0;
#         if (first1 && first2) {
#             // Handle initials vs full names
#             if ((first1.length === 1 && first2.length > 1) || (first2.length === 1 && first1.length > 1)) {
#                 const initial1 = first1.charAt(0);
#                 const initial2 = first2.charAt(0);
#                 firstNameScore = initial1 === initial2 ? 0.9 : 0; // Increased from 0.8 to 0.9
#             } else {
#                 firstNameScore = calculateStringSimilarity(first1, first2);
#             }
#         } else if (!first1 && !first2) {
#             firstNameScore = 1.0; // Both empty, perfect match
#         }
        
#         // Middle name similarity (more lenient)
#         let middleNameScore = 1.0; // Changed from 0.7 to 1.0 when both empty
#         if (middle1 && middle2) {
#             middleNameScore = calculateStringSimilarity(middle1, middle2);
#         } else if ((middle1 && !middle2) || (!middle1 && middle2)) {
#             middleNameScore = 0.8; // Increased penalty for missing middle name
#         }

#         // Check for perfect match
#         if (lastNameScore === 1.0 && firstNameScore === 1.0 && middleNameScore === 1.0) {
#             return 1.0;
#         }

#         // Enhanced weighted score calculation
#         let totalScore = 0;
        
#         // Last name is most important (65% weight)
#         totalScore += lastNameScore * 0.65;
        
#         // First name is important (30% weight)
#         totalScore += firstNameScore * 0.30;
        
#         // Middle name has small weight (5% weight)
#         totalScore += middleNameScore * 0.05;

#         return Math.min(1.0, totalScore);
#     }

#     /**
#      * Main enhanced name matching function
#      */
#     function enhancedMatchArtistNames(name1, name2) {
#         try {
#             if (!name1 || !name2 || name1.length === 0 || name2.length === 0) {
#                 return {
#                     jaro_winkler_score: 0.0,
#                     enhanced_score: 0.0,
#                     best_format_match: 0.0,
#                     normalized_match: 0.0
#                 };
#             }
            
#             // Clean and normalize inputs
#             let cleanName1 = name1.toString().trim().toUpperCase();
#             let cleanName2 = name2.toString().trim().toUpperCase();
            
#             // Calculate original Jaro-Winkler for backward compatibility
#             const jaroWinklerScore = jaroWinklerSimilarity(cleanName1, cleanName2);
            
#             // Check for direct match after normalization
#             const normalizedName1 = normalizeNameForComparison(name1);
#             const normalizedName2 = normalizeNameForComparison(name2);
            
#             let normalizedMatch = 0.0;
#             if (normalizedName1 === normalizedName2) {
#                 normalizedMatch = 1.0;
#             } else {
#                 normalizedMatch = calculateStringSimilarity(normalizedName1, normalizedName2);
#             }
            
#             // OPTION 1: ADD EXACT REVERSAL CHECK
#             // Check for exact reversal match (for 2-part and 3-part names)
#             const parts1 = normalizedName1.split(/\\s+/);
#             const parts2 = normalizedName2.split(/\\s+/);
            
#             // Handle 2-part names: "GORE MICHAEL" vs "MICHAEL GORE"
#             if (parts1.length === 2 && parts2.length === 2) {
#                 if (parts1[0] === parts2[1] && parts1[1] === parts2[0]) {
#                     return {
#                         jaro_winkler_score: jaroWinklerScore,
#                         enhanced_score: 1.0,
#                         best_format_match: 1.0,
#                         normalized_match: 1.0
#                     };
#                 }
#             }
            
#             // Handle 3-part names: "URIETA MARTIN SOLANO" vs "MARTIN URIETA SOLANO"
#             if (parts1.length === 3 && parts2.length === 3) {
#                 // Check if it's a simple reordering of the same three names
#                 const sorted1 = parts1.slice().sort();
#                 const sorted2 = parts2.slice().sort();
                
#                 if (sorted1.length === sorted2.length && 
#                     sorted1.every((val, index) => val === sorted2[index])) {
#                     return {
#                         jaro_winkler_score: jaroWinklerScore,
#                         enhanced_score: 1.0,
#                         best_format_match: 1.0,
#                         normalized_match: 1.0
#                     };
#                 }
#             }
            
#             // Handle general case: check if both names contain exactly the same words
#             if (parts1.length === parts2.length && parts1.length > 1) {
#                 const sorted1 = parts1.slice().sort();
#                 const sorted2 = parts2.slice().sort();
                
#                 if (sorted1.length === sorted2.length && 
#                     sorted1.every((val, index) => val === sorted2[index])) {
#                     return {
#                         jaro_winkler_score: jaroWinklerScore,
#                         enhanced_score: 1.0,
#                         best_format_match: 1.0,
#                         normalized_match: 1.0
#                     };
#                 }
#             }
            
#             // Parse names into different formats
#             const parsed1 = parseNameComponents(name1);
#             const parsed2 = parseNameComponents(name2);
            
#             // Try all format combinations and find the best match
#             let bestFormatScore = 0;
            
#             if (parsed1.formats && parsed2.formats) {
#                 for (const format1 of parsed1.formats) {
#                     for (const format2 of parsed2.formats) {
#                         const currentScore = calculateMatchScore(format1, format2);
#                         bestFormatScore = Math.max(bestFormatScore, currentScore);
#                     }
#                 }
#             }
            
#             // Calculate enhanced score combining multiple approaches
#             const enhancedScore = Math.max(
#                 jaroWinklerScore,
#                 normalizedMatch,
#                 bestFormatScore
#             );
            
#             return {
#                 jaro_winkler_score: jaroWinklerScore,
#                 enhanced_score: enhancedScore,
#                 best_format_match: bestFormatScore,
#                 normalized_match: normalizedMatch
#             };
            
#         } catch (e) {
#             return {
#                 jaro_winkler_score: 0.0,
#                 enhanced_score: 0.0,
#                 best_format_match: 0.0,
#                 normalized_match: 0.0
#             };
#         }
#     }

#     return enhancedMatchArtistNames(NAME1, NAME2);
#     $$;
#     """

#     session.sql(create_udf_sql).collect()
#     print("Enhanced JavaScript UDF for artist name matching created successfully.")


# def create_unified_match_table(session, work_ids=None):
   
    
#     # Build WHERE clause for work filtering - FIXED LOGIC
#     mode_description = "ALL WORKS"
    
#     if work_ids:
#         work_ids_str = "', '".join(work_ids)
#         work_ids_formatted = f"'{work_ids_str}'"
#         where_clause_step1 = f"WHERE APRA_WORK_ID IN ({work_ids_formatted})"
#         where_clause_step2 = f"WHERE APRA_WORK_ID IN ({work_ids_formatted}) AND MUZOOKA_TRACK_ID IS NOT NULL"
#         mode_description = f"SPECIFIC WORKS: {work_ids}"
#     else:
#         where_clause_step1 = ""  # No filtering for full mode
#         where_clause_step2 = "WHERE MUZOOKA_TRACK_ID IS NOT NULL"  # Only filter out NULL tracks
    
#     print(f"Creating unified match table for {mode_description}...")
#     print("Goal: Exactly one row per unique ARTIST_NAME × MUZOOKA_TRACK_ID combination")
    
#     # Single comprehensive query that creates the complete result set
#     unified_sql = f"""
#     WITH 
#     -- Step 1: Get DISTINCT artist names per work (deduplicate by ARTIST_NAME)
#     unique_artist_names AS (
#         SELECT DISTINCT
#             APRA_WORK_ID,
#             APRA_ARTIST_NAME,
#             -- Get any associated metadata for this artist name
#             MIN(APRA_ARTIST_ID) AS APRA_ARTIST_ID,  -- Use MIN to get one ID per name
#             MIN(RDC_WORKS_ID) AS RDC_WORKS_ID,
#             MIN(APRA_ARTIST_CT) AS APRA_ARTIST_CT,
#             MIN(APRA_TOTAL_ARTISTS) AS APRA_TOTAL_ARTISTS
#         FROM ADC_ARTIST_DELIMITER_COUNT_CLEAN
#         {where_clause_step1}
#         GROUP BY APRA_WORK_ID, APRA_ARTIST_NAME
#     ),
    
#     -- Step 2: Get unique tracks per work - FIXED WHERE CLAUSE
#     unique_tracks AS (
#         SELECT DISTINCT
#             APRA_WORK_ID,
#             MUZOOKA_TRACK_ID
#         FROM ADC_ARTIST_DELIMITER_COUNT_CLEAN
#         {where_clause_step2}
#     ),
    
#     -- Step 3: Get Muzooka track metadata
#     track_metadata AS (
#         SELECT DISTINCT
#             MUZOOKA_TRACK_ID,
#             MZK_TOTAL_ARTISTS
#         FROM MZK_ARTIST_DELIMITER_COUNT_CLEAN
#     ),
    
#     -- Step 4: Create COMPLETE CARTESIAN PRODUCT (every unique artist name x every track per work)
#     complete_combinations AS (
#         SELECT 
#             a.APRA_WORK_ID,
#             a.APRA_ARTIST_ID,
#             a.RDC_WORKS_ID,
#             a.APRA_ARTIST_NAME,
#             a.APRA_ARTIST_CT,
#             a.APRA_TOTAL_ARTISTS,
#             t.MUZOOKA_TRACK_ID,
#             COALESCE(tm.MZK_TOTAL_ARTISTS, 0) AS MZK_TOTAL_ARTISTS
#         FROM unique_artist_names a
#         CROSS JOIN unique_tracks t
#         LEFT JOIN track_metadata tm ON t.MUZOOKA_TRACK_ID = tm.MUZOOKA_TRACK_ID
#         WHERE a.APRA_WORK_ID = t.APRA_WORK_ID
#     ),
    
#     -- Step 5: Calculate name match scores for each combination
#     artist_track_matches AS (
#         SELECT 
#             cc.APRA_WORK_ID,
#             cc.APRA_ARTIST_ID,
#             cc.MUZOOKA_TRACK_ID,
#             cc.APRA_ARTIST_NAME,
#             -- Calculate best match score for this artist against all Muzooka artists on this track
#             COALESCE(MAX(ENHANCED_MATCH_ARTIST_NAMES(cc.APRA_ARTIST_NAME, m.MUZOOKA_ARTIST_NAME):enhanced_score::FLOAT), 0.0) AS NAME_MATCH_SCORE
#         FROM complete_combinations cc
#         LEFT JOIN MZK_ARTIST_DELIMITER_COUNT_CLEAN m
#         ON cc.MUZOOKA_TRACK_ID = m.MUZOOKA_TRACK_ID
#         GROUP BY cc.APRA_WORK_ID, cc.APRA_ARTIST_ID, cc.MUZOOKA_TRACK_ID, cc.APRA_ARTIST_NAME
#     ),
    
#     -- Step 6: Calculate writer match percentage at work-track level
    
#     work_track_writer_percentage AS (
#         SELECT 
#             atm.APRA_WORK_ID,
#             atm.MUZOOKA_TRACK_ID,
#             SUM(atm.NAME_MATCH_SCORE) AS total_match_score,
#             MAX(cc.APRA_TOTAL_ARTISTS) AS work_total_artists,
           
#             CASE 
#                 WHEN MAX(cc.APRA_TOTAL_ARTISTS) > 0 THEN
#                     SUM(atm.NAME_MATCH_SCORE) / MAX(cc.APRA_TOTAL_ARTISTS)
#                 ELSE 0.0
#             END AS ARTIST_MTCH_PCNTG
#         FROM artist_track_matches atm
#         JOIN complete_combinations cc 
#             ON atm.APRA_WORK_ID = cc.APRA_WORK_ID 
#             AND atm.MUZOOKA_TRACK_ID = cc.MUZOOKA_TRACK_ID
#         GROUP BY atm.APRA_WORK_ID, atm.MUZOOKA_TRACK_ID  --  GROUPED BY WORK + TRACK
#     )
    
#     -- Step 7: Final result set
#     SELECT
#         cc.RDC_WORKS_ID,
#         cc.APRA_ARTIST_ID,
#         cc.APRA_WORK_ID,
#         cc.MUZOOKA_TRACK_ID,
#         cc.APRA_ARTIST_NAME,
#         cc.APRA_TOTAL_ARTISTS,
#         cc.MZK_TOTAL_ARTISTS,
#         COUNT(DISTINCT cc.APRA_ARTIST_NAME) OVER (PARTITION BY cc.APRA_WORK_ID) AS APRA_MATCHED_TOTAL_ARTISTS,
#         cc.APRA_ARTIST_CT,
#         COALESCE(atm.NAME_MATCH_SCORE, 0) AS NAME_MATCH_SCORE,
#         ROUND(COALESCE(wtp.ARTIST_MTCH_PCNTG, 0), 6) AS ARTIST_MTCH_PCNTG
#     FROM 
#         complete_combinations cc
#     LEFT JOIN 
#         artist_track_matches atm
#     ON cc.APRA_WORK_ID = atm.APRA_WORK_ID 
#        AND cc.APRA_ARTIST_ID = atm.APRA_ARTIST_ID 
#        AND cc.MUZOOKA_TRACK_ID = atm.MUZOOKA_TRACK_ID
#     LEFT JOIN 
#         work_track_writer_percentage wtp
#     ON cc.APRA_WORK_ID = wtp.APRA_WORK_ID 
#        AND cc.MUZOOKA_TRACK_ID = wtp.MUZOOKA_TRACK_ID
#     ORDER BY
#         cc.APRA_WORK_ID, cc.APRA_ARTIST_NAME, cc.MUZOOKA_TRACK_ID
#     """
    
#     # Execute and create the results table
#     results = session.sql(unified_sql)
    
#     # Choose table name based on mode
#     table_name = "ARTIST_NAME_MATCH_UNIFIED_FULL" if not work_ids else "ARTIST_NAME_MATCH_UNIFIED_SPECIFIC"
    
#     results.write.mode("overwrite").save_as_table(table_name)
    
#     print(f" Table {table_name} created successfully!")
#     print(f"Processing mode: {mode_description}")
    
#     # Rest of the function remains the same...
#     return table_name

# def run_specific_works_pipeline(session, work_ids):
#     """Run pipeline for specific work IDs using unified implementation"""
    
#     if not work_ids:
#         print("No work IDs provided.")
#         return
    
#     print(f"Running UNIFIED artist matching for work IDs: {work_ids}")
    
#     # Create UDF first
#     create_enhanced_udf(session)
    
#     # Run unified pipeline for specific work IDs
#     table_name = create_unified_match_table(session, work_ids)
    
#     # Show detailed results for specific works if requested
#     if 'GW39809338' in work_ids:
#         print(f"\n=== DETAILED RESULTS FOR GW39809338 ===")
#         detailed_sql = f"""
#         SELECT 
#             APRA_WORK_ID,
#             MUZOOKA_TRACK_ID,
#             APRA_ARTIST_NAME,
#             NAME_MATCH_SCORE,
#             ARTIST_MTCH_PCNTG
#         FROM {table_name}
#         WHERE APRA_WORK_ID = 'GW39809338'
#         ORDER BY MUZOOKA_TRACK_ID, APRA_ARTIST_NAME
#         """
        
#         detailed_results = session.sql(detailed_sql)
#         detailed_results.show()
        
#         # Show artist and track breakdown
#         print(f"\n=== ARTISTS AND TRACKS FOR GW39809338 ===")
#         breakdown_sql = f"""
#         SELECT 'Artists' as type, APRA_ARTIST_NAME as name, APRA_ARTIST_ID as id
#         FROM {table_name}
#         WHERE APRA_WORK_ID = 'GW39809338'
#         GROUP BY APRA_ARTIST_NAME, APRA_ARTIST_ID
        
#         UNION ALL
        
#         SELECT 'Tracks' as type, MUZOOKA_TRACK_ID as name, NULL as id
#         FROM {table_name}
#         WHERE APRA_WORK_ID = 'GW39809338'
#         GROUP BY MUZOOKA_TRACK_ID
        
#         ORDER BY type, name
#         """
        
#         breakdown_results = session.sql(breakdown_sql)
#         breakdown_results.show()
    
#     print(f"\n UNIFIED PIPELINE COMPLETED FOR SPECIFIC WORKS!")
#     print(f" Table created: {table_name}")
    
# def run_full_pipeline(session):
#     """Run the complete pipeline for all works using unified implementation"""
    
#     print("Starting UNIFIED artist name matching pipeline for ALL WORKS...")
    
#     # Create UDF first
#     create_enhanced_udf(session)
    
#     # Run unified pipeline for all works
#     table_name = create_unified_match_table(session, work_ids=None)
    
#     print(f"\n UNIFIED PIPELINE COMPLETED FOR ALL WORKS!")
#     print(f" Table created: {table_name}")
  
# def main(session):
#     """
#     Main function with unified implementation for both modes
#     """
    
#     # Set run mode and parameters
#     run_mode = 'full'  # Options: 'full', 'specific_works'
    
#     # Define specific work IDs for testing
#     specific_work_ids = [
#         'GW39809338',      # The work mentioned in the question
#         # 'GW54087724',      
#         # 'GW45739725',  
#         # 'GW71382587',
#         # 'GW63248503',
#         'GW71382587'
#     ]
    
#     if run_mode.lower() == 'full':
#         # Run unified pipeline for all data
#         run_full_pipeline(session)
        
#     elif run_mode.lower() == 'specific_works':
#         # Run unified pipeline for specific work IDs
#         run_specific_works_pipeline(session, specific_work_ids)
        
#     else:
#         print(f"Invalid run mode: {run_mode}. Please specify 'full' or 'specific_works'")
#         return
    
#     print(f"\n UNIFIED ARTIST NAME MATCHING COMPLETED!")
#     print(f"Mode: {run_mode}")
   
#     if run_mode.lower() == 'specific_works':
#         print(f"Processed work IDs: {specific_work_ids}")


# if __name__ == "__main__":
#     main(session)

In [ ]:

#better part score handling


# def create_enhanced_udf(session):
#     """Create enhanced JavaScript UDF for artist name matching with multiple format handling"""
    
#     create_udf_sql = """
#     CREATE OR REPLACE FUNCTION ENHANCED_MATCH_ARTIST_NAMES(name1 STRING, name2 STRING)
#     RETURNS OBJECT
#     LANGUAGE JAVASCRIPT
#     AS
#     $$
    
#     /**
#      * Normalize names to handle special cases:
#      * 1. Remove/normalize generational suffixes
#      * 2. Replace hyphens with spaces to handle hyphenated names
#      * 3. Normalize special prefixes
#      * 4. Remove bracket content
#      */
#     function normalizeNameForComparison(name) {
#         if (!name) return "";
        
#         // Remove content in brackets first
#         name = name.replace(/\\([^\\)]*\\)/g, "");
        
#         name = name.toUpperCase().trim();
        
#         // 1. Remove generational suffixes: I, II, III, IV, V and their variations
#         name = name.replace(/\\b(I{1,3}|IV|V|1ST|2ND|3RD|[4-9]TH)\\b$/g, "").trim();
        
#         // 2. Replace hyphens with spaces for better matching
#         name = name.replace(/-/g, " ");
        
#         // 3. Special handling for prefixes - ensure consistent spacing
#         const special_prefixes = ["VAN", "VON", "DE", "DER", "LA", "LE", "DI", "DEL", "DOS", "DA", "DU", "AL", "EL", "JESUS", "CHAVEZ"];
        
#         // Create regex patterns for each prefix to match only when it's a standalone word
#         for (const prefix of special_prefixes) {
#             const regex = new RegExp(`\\\\b${prefix}\\\\b`, 'g');
#             // Ensure consistent spacing for each prefix
#             name = name.replace(regex, prefix);
#         }
        
#         // Remove extra whitespace
#         name = name.replace(/\\s+/g, " ").trim();
        
#         return name;
#     }

#     /**
#      * Enhanced name parsing that handles multiple formats
#      */
#     function parseNameComponents(name) {
#         if (!name || name === "") {
#             return {formats: []};
#         }

#         // Normalize the name first
#         name = normalizeNameForComparison(name);

#         // Special suffixes and prefixes
#         const suffixes = ["SENIOR", "SNR", "SR", "JUNIOR", "JNR", "JR"];
#         const special_prefixes = ["VAN", "VON", "DE", "DER", "LA", "LE", "DI", "DEL", "DOS", "DA", "DU", "AL", "EL", "JESUS", "CHAVEZ"];

#         // Clean and split the name
#         let parts = name.split(/\\s+/);

#         // Handle name with no spaces
#         if (parts.length === 1) {
#             return {formats: [{first: "", last: parts[0], middle: "", suffix: "", is_initial: false}]};
#         }

#         // Check for and extract suffixes
#         let suffix = "";
#         for (const suffixTerm of suffixes) {
#             const suffixIndex = parts.findIndex(part => part === suffixTerm);
#             if (suffixIndex !== -1) {
#                 suffix = parts[suffixIndex];
#                 parts.splice(suffixIndex, 1);
#                 break;
#             }
#         }

#         // After removing suffix, check if we're back to a single name
#         if (parts.length === 1) {
#             return {formats: [{first: "", last: parts[0], middle: "", suffix: suffix, is_initial: false}]};
#         }

#         // Generate multiple format possibilities
#         const formatsToCheck = [];
        
#         // Check for special multi-word last names with prefixes like "VAN BEETHOVEN"
#         let hasSpecialPrefix = false;
#         let specialPrefixIndex = -1;
        
#         for (let i = 0; i < parts.length; i++) {
#             if (special_prefixes.includes(parts[i])) {
#                 hasSpecialPrefix = true;
#                 specialPrefixIndex = i;
#                 break;
#             }
#         }
        
#         if (hasSpecialPrefix) {
#             if (specialPrefixIndex === 0 && parts.length >= 2) {
#                 // Case: "VAN BEETHOVEN" or "VAN BEETHOVEN LUDWIG"
#                 if (parts.length === 2) {
#                     // Just "VAN BEETHOVEN" - treat as compound last name
#                     formatsToCheck.push({
#                         first: "",
#                         last: parts.join(" "),
#                         middle: "",
#                         suffix: suffix,
#                         is_initial: false
#                     });
#                 } else {
#                     // "BEETHOVEN LUDWIG VAN" - LASTNAME FIRSTNAME PREFIX format
#                     formatsToCheck.push({
#                         first: parts[1],
#                         last: parts[0] + " " + parts[2],
#                         middle: parts.slice(3).join(" "),
#                         suffix: suffix,
#                         is_initial: parts[1] && parts[1].length === 1
#                     });
                    
#                     // "VAN BEETHOVEN LUDWIG" - PREFIX LASTNAME FIRSTNAME format
#                     const lastName = parts.slice(0, 2).join(" ");
#                     const firstName = parts[2];
#                     const middleName = parts.slice(3).join(" ");
                    
#                     formatsToCheck.push({
#                         first: firstName,
#                         last: lastName,
#                         middle: middleName,
#                         suffix: suffix,
#                         is_initial: firstName && firstName.length === 1
#                     });
#                 }
#             } else if (specialPrefixIndex > 0) {
#                 // Case: "LUDWIG VAN BEETHOVEN" - FIRSTNAME PREFIX LASTNAME format
#                 const firstName = parts.slice(0, specialPrefixIndex).join(" ");
#                 const lastName = parts.slice(specialPrefixIndex).join(" ");
                
#                 formatsToCheck.push({
#                     first: firstName,
#                     last: lastName,
#                     middle: "",
#                     suffix: suffix,
#                     is_initial: firstName.length === 1
#                 });
                
#                 // Also try reversed: LASTNAME FIRSTNAME format (VAN BEETHOVEN LUDWIG)
#                 formatsToCheck.push({
#                     first: firstName,
#                     last: lastName,
#                     middle: "",
#                     suffix: suffix,
#                     is_initial: firstName.length === 1
#                 });
#             }
#         }
        
#         // Standard format handling for all cases
#         if (parts.length === 2) {
#             // Two parts: ALWAYS try both orders for cases like "BERNIE BEN" vs "BEN BERNIE"
#             formatsToCheck.push({
#                 first: parts[0],
#                 last: parts[1],
#                 middle: "",
#                 suffix: suffix,
#                 is_initial: parts[0].length === 1
#             });
            
#             formatsToCheck.push({
#                 first: parts[1],
#                 last: parts[0],
#                 middle: "",
#                 suffix: suffix,
#                 is_initial: parts[1].length === 1
#             });
#         } else if (parts.length >= 3) {
#             // Format 1: LASTNAME FIRSTNAME MIDDLENAME (ADC Style)
#             formatsToCheck.push({
#                 first: parts[1],
#                 last: parts[0],
#                 middle: parts.slice(2).join(" "),
#                 suffix: suffix,
#                 is_initial: parts[1].length === 1
#             });
            
#             // Format 2: FIRSTNAME MIDDLENAME LASTNAME (Muzooka Style)
#             formatsToCheck.push({
#                 first: parts[0],
#                 last: parts[parts.length - 1],
#                 middle: parts.slice(1, parts.length - 1).join(" "),
#                 suffix: suffix,
#                 is_initial: parts[0].length === 1
#             });
            
#             // Format 3: FIRSTNAME LASTNAME (treating middle as part of first)
#             formatsToCheck.push({
#                 first: parts.slice(0, parts.length - 1).join(" "),
#                 last: parts[parts.length - 1],
#                 middle: "",
#                 suffix: suffix,
#                 is_initial: false
#             });
            
#             // Format 4: LASTNAME FIRSTNAME (treating middle as part of last)
#             formatsToCheck.push({
#                 first: parts[parts.length - 1],
#                 last: parts.slice(0, parts.length - 1).join(" "),
#                 middle: "",
#                 suffix: suffix,
#                 is_initial: parts[parts.length - 1].length === 1
#             });
            
#             // Format 5: For compound last names (first two parts as last name)
#             if (parts.length >= 3) {
#                 formatsToCheck.push({
#                     first: parts.slice(2).join(" "),
#                     last: parts.slice(0, 2).join(" "),
#                     middle: "",
#                     suffix: suffix,
#                     is_initial: false
#                 });
#             }
            
#             // Format 6: For compound last names (last two parts as last name)
#             if (parts.length >= 3) {
#                 formatsToCheck.push({
#                     first: parts.slice(0, parts.length - 2).join(" "),
#                     last: parts.slice(parts.length - 2).join(" "),
#                     middle: "",
#                     suffix: suffix,
#                     is_initial: false
#                 });
#             }
#         }

#         return {formats: formatsToCheck};
#     }

#     /**
#      * Calculate string similarity using Levenshtein distance
#      */
#     function calculateStringSimilarity(s1, s2) {
#         if (!s1 || !s2) return 0;
#         if (s1 === s2) return 1.0;

#         const str1 = s1.toUpperCase().trim();
#         const str2 = s2.toUpperCase().trim();

#         if (str1 === str2) return 1.0;
#         if (str1 === "" || str2 === "") return 0.0;

#         const len1 = str1.length;
#         const len2 = str2.length;
#         const matrix = Array(len1 + 1).fill().map(() => Array(len2 + 1).fill(0));

#         for (let i = 0; i <= len1; i++) matrix[i][0] = i;
#         for (let j = 0; j <= len2; j++) matrix[0][j] = j;

#         for (let i = 1; i <= len1; i++) {
#             for (let j = 1; j <= len2; j++) {
#                 const cost = str1[i-1] === str2[j-1] ? 0 : 1;
#                 matrix[i][j] = Math.min(
#                     matrix[i-1][j] + 1,
#                     matrix[i][j-1] + 1,
#                     matrix[i-1][j-1] + cost
#                 );
#             }
#         }

#         const distance = matrix[len1][len2];
#         const maxLen = Math.max(len1, len2);
#         return 1 - (distance / maxLen);
#     }

#     /**
#      * Jaro-Winkler similarity implementation
#      */
#     function jaroWinklerSimilarity(s1, s2) {
#         if (!s1 || !s2 || s1.length === 0 || s2.length === 0) return 0;
#         if (s1 === s2) return 1;
        
#         let m = 0;
#         let t = 0;
#         let range = Math.floor(Math.max(s1.length, s2.length) / 2) - 1;
#         range = Math.max(0, range);
        
#         let s1Matches = new Array(s1.length).fill(false);
#         let s2Matches = new Array(s2.length).fill(false);
        
#         for (let i = 0; i < s1.length; i++) {
#             let start = Math.max(0, i - range);
#             let end = Math.min(i + range + 1, s2.length);
            
#             for (let j = start; j < end; j++) {
#                 if (!s2Matches[j] && s1[i] === s2[j]) {
#                     s1Matches[i] = true;
#                     s2Matches[j] = true;
#                     m++;
#                     break;
#                 }
#             }
#         }
        
#         if (m === 0) return 0;
        
#         let k = 0;
#         for (let i = 0; i < s1.length; i++) {
#             if (s1Matches[i]) {
#                 while (k < s2.length && !s2Matches[k]) k++;
#                 if (k < s2.length && s1[i] !== s2[k]) t++;
#                 if (k < s2.length) k++;
#             }
#         }
        
#         t = Math.floor(t / 2);
#         let jaroSim = (m / s1.length + m / s2.length + (m - t) / m) / 3;
        
#         let p = 0.1;
#         let l = 0;
        
#         for (let i = 0; i < Math.min(4, Math.min(s1.length, s2.length)); i++) {
#             if (s1[i] === s2[i]) {
#                 l++;
#             } else {
#                 break;
#             }
#         }
        
#         return jaroSim + (l * p * (1 - jaroSim));
#     }

#     /**
#      * Calculate match score between two name formats with enhanced logic
#      */
#     function calculateMatchScore(format1, format2) {
#         if (!format1 || !format2) return 0;

#         const first1 = format1.first || "";
#         const last1 = format1.last || "";
#         const middle1 = format1.middle || "";
        
#         const first2 = format2.first || "";
#         const last2 = format2.last || "";
#         const middle2 = format2.middle || "";

#         // Last name similarity (most important)
#         const lastNameScore = calculateStringSimilarity(last1, last2);
        
#         // First name similarity with enhanced initial handling
#         let firstNameScore = 0;
#         if (first1 && first2) {
#             // Handle initials vs full names
#             if ((first1.length === 1 && first2.length > 1) || (first2.length === 1 && first1.length > 1)) {
#                 const initial1 = first1.charAt(0);
#                 const initial2 = first2.charAt(0);
#                 firstNameScore = initial1 === initial2 ? 0.9 : 0; // Increased from 0.8 to 0.9
#             } else {
#                 firstNameScore = calculateStringSimilarity(first1, first2);
#             }
#         } else if (!first1 && !first2) {
#             firstNameScore = 1.0; // Both empty, perfect match
#         }
        
#         // Middle name similarity (more lenient)
#         let middleNameScore = 1.0; // Changed from 0.7 to 1.0 when both empty
#         if (middle1 && middle2) {
#             middleNameScore = calculateStringSimilarity(middle1, middle2);
#         } else if ((middle1 && !middle2) || (!middle1 && middle2)) {
#             middleNameScore = 0.8; // Increased penalty for missing middle name
#         }

#         // Check for perfect match
#         if (lastNameScore === 1.0 && firstNameScore === 1.0 && middleNameScore === 1.0) {
#             return 1.0;
#         }

#         // Enhanced weighted score calculation
#         let totalScore = 0;
        
#         // Last name is most important (65% weight)
#         totalScore += lastNameScore * 0.65;
        
#         // First name is important (30% weight)
#         totalScore += firstNameScore * 0.30;
        
#         // Middle name has small weight (5% weight)
#         totalScore += middleNameScore * 0.05;

#         return Math.min(1.0, totalScore);
#     }

#     /**
#      * Main enhanced name matching function
#      */
#     function enhancedMatchArtistNames(name1, name2) {
#         try {
#             if (!name1 || !name2 || name1.length === 0 || name2.length === 0) {
#                 return {
#                     jaro_winkler_score: 0.0,
#                     enhanced_score: 0.0,
#                     best_format_match: 0.0,
#                     normalized_match: 0.0
#                 };
#             }
            
#             // Clean and normalize inputs
#             let cleanName1 = name1.toString().trim().toUpperCase();
#             let cleanName2 = name2.toString().trim().toUpperCase();
            
#             // Calculate original Jaro-Winkler for backward compatibility
#             const jaroWinklerScore = jaroWinklerSimilarity(cleanName1, cleanName2);
            
#             // Check for direct match after normalization
#             const normalizedName1 = normalizeNameForComparison(name1);
#             const normalizedName2 = normalizeNameForComparison(name2);
            
#             let normalizedMatch = 0.0;
#             if (normalizedName1 === normalizedName2) {
#                 normalizedMatch = 1.0;
#             } else {
#                 normalizedMatch = calculateStringSimilarity(normalizedName1, normalizedName2);
#             }
            
#             // OPTION 1: ADD EXACT REVERSAL CHECK
#             // Check for exact reversal match (for 2-part and 3-part names)
#             const parts1 = normalizedName1.split(/\\s+/);
#             const parts2 = normalizedName2.split(/\\s+/);
            
#             // Handle 2-part names: "GORE MICHAEL" vs "MICHAEL GORE"
#             if (parts1.length === 2 && parts2.length === 2) {
#                 if (parts1[0] === parts2[1] && parts1[1] === parts2[0]) {
#                     return {
#                         jaro_winkler_score: jaroWinklerScore,
#                         enhanced_score: 1.0,
#                         best_format_match: 1.0,
#                         normalized_match: 1.0
#                     };
#                 }
#             }
            
#             // Handle 3-part names: "URIETA MARTIN SOLANO" vs "MARTIN URIETA SOLANO"
#             if (parts1.length === 3 && parts2.length === 3) {
#                 // Check if it's a simple reordering of the same three names
#                 const sorted1 = parts1.slice().sort();
#                 const sorted2 = parts2.slice().sort();
                
#                 if (sorted1.length === sorted2.length && 
#                     sorted1.every((val, index) => val === sorted2[index])) {
#                     return {
#                         jaro_winkler_score: jaroWinklerScore,
#                         enhanced_score: 1.0,
#                         best_format_match: 1.0,
#                         normalized_match: 1.0
#                     };
#                 }
#             }
            
#             // Handle general case: check if both names contain exactly the same words
#             if (parts1.length === parts2.length && parts1.length > 1) {
#                 const sorted1 = parts1.slice().sort();
#                 const sorted2 = parts2.slice().sort();
                
#                 if (sorted1.length === sorted2.length && 
#                     sorted1.every((val, index) => val === sorted2[index])) {
#                     return {
#                         jaro_winkler_score: jaroWinklerScore,
#                         enhanced_score: 1.0,
#                         best_format_match: 1.0,
#                         normalized_match: 1.0
#                     };
#                 }
#             }
            
#             // Parse names into different formats
#             const parsed1 = parseNameComponents(name1);
#             const parsed2 = parseNameComponents(name2);
            
#             // Try all format combinations and find the best match
#             let bestFormatScore = 0;
            
#             if (parsed1.formats && parsed2.formats) {
#                 for (const format1 of parsed1.formats) {
#                     for (const format2 of parsed2.formats) {
#                         const currentScore = calculateMatchScore(format1, format2);
#                         bestFormatScore = Math.max(bestFormatScore, currentScore);
#                     }
#                 }
#             }
            
#             // Calculate enhanced score combining multiple approaches
#             const enhancedScore = Math.max(
#                 jaroWinklerScore,
#                 normalizedMatch,
#                 bestFormatScore
#             );
            
#             return {
#                 jaro_winkler_score: jaroWinklerScore,
#                 enhanced_score: enhancedScore,
#                 best_format_match: bestFormatScore,
#                 normalized_match: normalizedMatch
#             };
            
#         } catch (e) {
#             return {
#                 jaro_winkler_score: 0.0,
#                 enhanced_score: 0.0,
#                 best_format_match: 0.0,
#                 normalized_match: 0.0
#             };
#         }
#     }

#     return enhancedMatchArtistNames(NAME1, NAME2);
#     $$;
#     """

#     session.sql(create_udf_sql).collect()
#     print("Enhanced JavaScript UDF for artist name matching created successfully.")


# def create_part_by_part_match_table(session, work_ids=None):
#     """Create unified match table with part-by-part matching logic"""
    
#     mode_description = "ALL WORKS"
    
#     if work_ids:
#         work_ids_str = "', '".join(work_ids)
#         work_ids_formatted = f"'{work_ids_str}'"
#         where_clause_step1 = f"WHERE APRA_WORK_ID IN ({work_ids_formatted})"
#         where_clause_step2 = f"WHERE APRA_WORK_ID IN ({work_ids_formatted}) AND MUZOOKA_TRACK_ID IS NOT NULL"
#         mode_description = f"SPECIFIC WORKS: {work_ids}"
#     else:
#         where_clause_step1 = ""  # No filtering for full mode
#         where_clause_step2 = "WHERE MUZOOKA_TRACK_ID IS NOT NULL"  # Only filter out NULL tracks
    
#     print(f"Creating part-by-part match table for {mode_description}...")
#     print("Goal: Compare each ADC delimited part against each MZK delimited part")
    
#     # Part-by-part matching query
#     unified_sql = f"""
#     WITH 
#     -- Step 1: Get unique artist entries with their delimited parts
#     unique_artist_names AS (
#         SELECT DISTINCT
#             APRA_WORK_ID,
#             APRA_ARTIST_NAME,
#             DELIMITED_PARTS AS ADC_DELIMITED_PARTS,
#             MIN(APRA_ARTIST_ID) AS APRA_ARTIST_ID,
#             MIN(RDC_WORKS_ID) AS RDC_WORKS_ID,
#             MIN(APRA_ARTIST_CT) AS APRA_ARTIST_CT,
#             MIN(APRA_TOTAL_ARTISTS) AS APRA_TOTAL_ARTISTS
#         FROM ADC_ARTIST_DELIMITER_COUNT_CLEAN
#         {where_clause_step1}
#         GROUP BY APRA_WORK_ID, APRA_ARTIST_NAME, DELIMITED_PARTS
#     ),
    
#     -- Step 2: Get unique tracks per work
#     unique_tracks AS (
#         SELECT DISTINCT
#             APRA_WORK_ID,
#             MUZOOKA_TRACK_ID
#         FROM ADC_ARTIST_DELIMITER_COUNT_CLEAN
#         {where_clause_step2}
#     ),
    
#     -- Step 3: Get Muzooka track metadata with delimited parts
#     track_metadata AS (
#         SELECT DISTINCT
#             MUZOOKA_TRACK_ID,
#             DELIMITED_PARTS AS MZK_DELIMITED_PARTS,
#             MZK_TOTAL_ARTISTS
#         FROM MZK_ARTIST_DELIMITER_COUNT_CLEAN
#     ),
    
#     -- Step 4: Create base combinations (APRA artist x Track)
#     base_combinations AS (
#         SELECT 
#             a.APRA_WORK_ID,
#             a.APRA_ARTIST_ID,
#             a.RDC_WORKS_ID,
#             a.APRA_ARTIST_NAME,
#             a.ADC_DELIMITED_PARTS,
#             a.APRA_ARTIST_CT,
#             a.APRA_TOTAL_ARTISTS,
#             t.MUZOOKA_TRACK_ID,
#             COALESCE(MAX(tm.MZK_TOTAL_ARTISTS), 0) AS MZK_TOTAL_ARTISTS
#         FROM unique_artist_names a
#         CROSS JOIN unique_tracks t
#         LEFT JOIN track_metadata tm ON t.MUZOOKA_TRACK_ID = tm.MUZOOKA_TRACK_ID
#         WHERE a.APRA_WORK_ID = t.APRA_WORK_ID
#         GROUP BY a.APRA_WORK_ID, a.APRA_ARTIST_ID, a.RDC_WORKS_ID, a.APRA_ARTIST_NAME, 
#                  a.ADC_DELIMITED_PARTS, a.APRA_ARTIST_CT, a.APRA_TOTAL_ARTISTS, t.MUZOOKA_TRACK_ID
#     ),
    
#     -- Step 5: Explode ADC delimited parts into individual parts
#     adc_parts_exploded AS (
#         SELECT 
#             bc.*,
#             adc_part.VALUE::STRING AS ADC_INDIVIDUAL_PART,
#             adc_part.INDEX AS ADC_PART_INDEX
#         FROM base_combinations bc,
#         LATERAL FLATTEN(PARSE_JSON(bc.ADC_DELIMITED_PARTS)) adc_part
#     ),
    
#     -- Step 6: Get MZK delimited parts for each track and explode them
#     mzk_parts_exploded AS (
#         SELECT 
#             tm.MUZOOKA_TRACK_ID,
#             tm.MZK_DELIMITED_PARTS,
#             mzk_part.VALUE::STRING AS MZK_INDIVIDUAL_PART,
#             mzk_part.INDEX AS MZK_PART_INDEX
#         FROM track_metadata tm,
#         LATERAL FLATTEN(PARSE_JSON(tm.MZK_DELIMITED_PARTS)) mzk_part
#     ),
    
#     -- Step 7: Create all combinations of ADC parts x MZK parts for each track
#     part_combinations AS (
#         SELECT 
#             ape.*,
#             mpe.MZK_DELIMITED_PARTS,
#             mpe.MZK_INDIVIDUAL_PART,
#             mpe.MZK_PART_INDEX,
#             -- Calculate match score between individual parts
#             ENHANCED_MATCH_ARTIST_NAMES(ape.ADC_INDIVIDUAL_PART, mpe.MZK_INDIVIDUAL_PART):enhanced_score::FLOAT AS PART_MATCH_SCORE
#         FROM adc_parts_exploded ape
#         LEFT JOIN mzk_parts_exploded mpe 
#         ON ape.MUZOOKA_TRACK_ID = mpe.MUZOOKA_TRACK_ID
#     ),
    
#     -- Step 8: Find best matching MZK part for each ADC part
#     best_part_matches AS (
#         SELECT 
#             APRA_WORK_ID,
#             APRA_ARTIST_ID,
#             MUZOOKA_TRACK_ID,
#             APRA_ARTIST_NAME,
#             ADC_INDIVIDUAL_PART,
#             ADC_PART_INDEX,
#             MAX(PART_MATCH_SCORE) AS BEST_PART_SCORE
#         FROM part_combinations
#         GROUP BY APRA_WORK_ID, APRA_ARTIST_ID, MUZOOKA_TRACK_ID, APRA_ARTIST_NAME, ADC_INDIVIDUAL_PART, ADC_PART_INDEX
#     ),
    
#     -- Step 9: Get the MZK delimited parts for the track that had the best overall matching
#     best_track_mzk_parts AS (
#         SELECT 
#             pc.APRA_WORK_ID,
#             pc.APRA_ARTIST_ID,
#             pc.MUZOOKA_TRACK_ID,
#             pc.APRA_ARTIST_NAME,
#             pc.MZK_DELIMITED_PARTS,
#             SUM(bpm.BEST_PART_SCORE) AS TOTAL_PART_MATCH_SCORE,
#             COUNT(DISTINCT pc.ADC_PART_INDEX) AS ADC_PARTS_COUNT
#         FROM part_combinations pc
#         INNER JOIN best_part_matches bpm 
#         ON pc.APRA_WORK_ID = bpm.APRA_WORK_ID 
#         AND pc.APRA_ARTIST_ID = bpm.APRA_ARTIST_ID 
#         AND pc.MUZOOKA_TRACK_ID = bpm.MUZOOKA_TRACK_ID
#         AND pc.ADC_INDIVIDUAL_PART = bpm.ADC_INDIVIDUAL_PART
#         AND pc.PART_MATCH_SCORE = bpm.BEST_PART_SCORE
#         GROUP BY pc.APRA_WORK_ID, pc.APRA_ARTIST_ID, pc.MUZOOKA_TRACK_ID, pc.APRA_ARTIST_NAME, pc.MZK_DELIMITED_PARTS
#     ),
    
#     -- Step 10: Select the best MZK delimited parts per artist-track combination
#     final_best_matches AS (
#         SELECT 
#             *,
#             ROW_NUMBER() OVER (
#                 PARTITION BY APRA_WORK_ID, APRA_ARTIST_ID, MUZOOKA_TRACK_ID 
#                 ORDER BY TOTAL_PART_MATCH_SCORE DESC, MZK_DELIMITED_PARTS
#             ) as rn
#         FROM best_track_mzk_parts
#     ),
    
#     -- Step 11: Calculate artist match percentage at work-track level
#     work_track_artist_percentage AS (
#         SELECT 
#             fbm.APRA_WORK_ID,
#             fbm.MUZOOKA_TRACK_ID,
#             SUM(fbm.TOTAL_PART_MATCH_SCORE) AS work_track_total_score,
#             MAX(fbm.ADC_PARTS_COUNT) * MAX(bc.APRA_TOTAL_ARTISTS) AS max_possible_score,
#             CASE 
#                 WHEN MAX(bc.APRA_TOTAL_ARTISTS) > 0 THEN
#                     SUM(fbm.TOTAL_PART_MATCH_SCORE) / MAX(bc.APRA_TOTAL_ARTISTS)
#                 ELSE 0.0
#             END AS ARTIST_MTCH_PCNTG
#         FROM final_best_matches fbm
#         JOIN base_combinations bc 
#             ON fbm.APRA_WORK_ID = bc.APRA_WORK_ID 
#             AND fbm.MUZOOKA_TRACK_ID = bc.MUZOOKA_TRACK_ID
#         WHERE fbm.rn = 1
#         GROUP BY fbm.APRA_WORK_ID, fbm.MUZOOKA_TRACK_ID
#     )
    
#     -- Step 12: Final result set with part-by-part matching logic
#     SELECT
#         bc.RDC_WORKS_ID,
#         bc.APRA_ARTIST_ID,
#         bc.APRA_WORK_ID,
#         bc.MUZOOKA_TRACK_ID,
#         bc.APRA_ARTIST_NAME,
#         bc.ADC_DELIMITED_PARTS,
#         COALESCE(fbm.MZK_DELIMITED_PARTS, '[]') AS MZK_DELIMITED_PARTS,
#         bc.APRA_TOTAL_ARTISTS,
#         bc.MZK_TOTAL_ARTISTS,
#         COUNT(DISTINCT bc.APRA_ARTIST_NAME) OVER (PARTITION BY bc.APRA_WORK_ID) AS APRA_MATCHED_TOTAL_ARTISTS,
#         bc.APRA_ARTIST_CT,
#         COALESCE(fbm.TOTAL_PART_MATCH_SCORE, 0) AS PART_MATCH_SCORE,
#         CASE 
#             WHEN bc.APRA_ARTIST_CT > 0 THEN
#                 COALESCE(fbm.TOTAL_PART_MATCH_SCORE, 0) / bc.APRA_ARTIST_CT
#             ELSE 0.0
#         END AS NAME_MATCH_SCORE,
#         ROUND(COALESCE(wtap.ARTIST_MTCH_PCNTG, 0), 6) AS ARTIST_MTCH_PCNTG
#     FROM 
#         base_combinations bc
#     LEFT JOIN 
#         final_best_matches fbm
#     ON bc.APRA_WORK_ID = fbm.APRA_WORK_ID 
#        AND bc.APRA_ARTIST_ID = fbm.APRA_ARTIST_ID 
#        AND bc.MUZOOKA_TRACK_ID = fbm.MUZOOKA_TRACK_ID
#        AND fbm.rn = 1
#     LEFT JOIN 
#         work_track_artist_percentage wtap
#     ON bc.APRA_WORK_ID = wtap.APRA_WORK_ID 
#        AND bc.MUZOOKA_TRACK_ID = wtap.MUZOOKA_TRACK_ID
#     ORDER BY
#         bc.APRA_WORK_ID, bc.APRA_ARTIST_NAME, bc.MUZOOKA_TRACK_ID
#     """
    
#     # Execute and create the results table
#     results = session.sql(unified_sql)
    
#     # Choose table name based on mode
#     table_name = "ARTIST_PART_BY_PART_MATCH_FULL" if not work_ids else "ARTIST_PART_BY_PART_MATCH_SPECIFIC"
    
#     results.write.mode("overwrite").save_as_table(table_name)
    
#     print(f"✅ Table {table_name} created successfully!")
#     print(f"Processing mode: {mode_description}")
    
#     return table_name


# def run_specific_works_pipeline(session, work_ids):
#     """Run pipeline for specific work IDs using part-by-part matching"""
    
#     if not work_ids:
#         print("No work IDs provided.")
#         return
    
#     print(f"Running PART-BY-PART artist matching for work IDs: {work_ids}")
    
#     # Create UDF first
#     create_enhanced_udf(session)
    
#     # Run part-by-part pipeline for specific work IDs
#     table_name = create_part_by_part_match_table(session, work_ids)
    
#     # Show detailed results for specific works if requested
#     if 'GW39809338' in work_ids:
#         print(f"\n=== DETAILED PART-BY-PART RESULTS FOR GW39809338 ===")
#         detailed_sql = f"""
#         SELECT 
#             APRA_WORK_ID,
#             MUZOOKA_TRACK_ID,
#             APRA_ARTIST_NAME,
#             ADC_DELIMITED_PARTS,
#             MZK_DELIMITED_PARTS,
#             APRA_ARTIST_CT,
#             PART_MATCH_SCORE,
#             NAME_MATCH_SCORE,
#             ARTIST_MTCH_PCNTG
#         FROM {table_name}
#         WHERE APRA_WORK_ID = 'GW39809338'
#         ORDER BY MUZOOKA_TRACK_ID, APRA_ARTIST_NAME
#         """
        
#         detailed_results = session.sql(detailed_sql)
#         detailed_results.show()
        
#         # Show part-by-part breakdown
#         print(f"\n=== PART-BY-PART MATCHING BREAKDOWN FOR GW39809338 ===")
#         breakdown_sql = f"""
#         WITH part_details AS (
#             SELECT 
#                 APRA_ARTIST_NAME,
#                 ADC_DELIMITED_PARTS,
#                 MZK_DELIMITED_PARTS,
#                 PART_MATCH_SCORE,
#                 NAME_MATCH_SCORE,
#                 APRA_ARTIST_CT
#             FROM {table_name}
#             WHERE APRA_WORK_ID = 'GW39809338'
#         )
#         SELECT 
#             APRA_ARTIST_NAME,
#             ADC_DELIMITED_PARTS as "ADC Parts",
#             MZK_DELIMITED_PARTS as "MZK Parts", 
#             APRA_ARTIST_CT as "ADC Part Count",
#             PART_MATCH_SCORE as "Total Part Score",
#             NAME_MATCH_SCORE as "Normalized Score (Score/Count)"
#         FROM part_details
#         ORDER BY NAME_MATCH_SCORE DESC
#         """
        
#         breakdown_results = session.sql(breakdown_sql)
#         breakdown_results.show()
    
#     print(f"\n🎯 PART-BY-PART PIPELINE COMPLETED FOR SPECIFIC WORKS!")
#     print(f"📊 Table created: {table_name}")
    

# def run_full_pipeline(session):
#     """Run the complete pipeline for all works using part-by-part matching"""
    
#     print("Starting PART-BY-PART artist name matching pipeline for ALL WORKS...")
    
#     # Create UDF first
#     create_enhanced_udf(session)
    
#     # Run part-by-part pipeline for all works
#     table_name = create_part_by_part_match_table(session, work_ids=None)
    
#     print(f"\n🎯 PART-BY-PART PIPELINE COMPLETED FOR ALL WORKS!")
#     print(f"📊 Table created: {table_name}")
  

# def main(session):
#     """
#     Main function with part-by-part matching implementation
#     """
    
#     # Set run mode and parameters
#     run_mode = 'specific_works'  # Options: 'full', 'specific_works'
    
#     # Define specific work IDs for testing
#     specific_work_ids = [
#         'GW39809338',      # The work mentioned in the question
#         'GW71382587'
#     ]
    
#     if run_mode.lower() == 'full':
#         # Run part-by-part pipeline for all data
#         run_full_pipeline(session)
        
#     elif run_mode.lower() == 'specific_works':
#         # Run part-by-part pipeline for specific work IDs
#         run_specific_works_pipeline(session, specific_work_ids)
        
#     else:
#         print(f"Invalid run mode: {run_mode}. Please specify 'full' or 'specific_works'")
#         return
    
#     print(f"\n🎯 PART-BY-PART ARTIST NAME MATCHING COMPLETED!")
#     print(f"Mode: {run_mode}")
   
#     if run_mode.lower() == 'specific_works':
#         print(f"Processed work IDs: {specific_work_ids}")


# if __name__ == "__main__":
#     main(session)

In [ ]:
#better part score + last name handling


def create_enhanced_udf(session):
    """Create enhanced JavaScript UDF for artist name matching with multiple format handling"""
    
    create_udf_sql = """
    CREATE OR REPLACE FUNCTION ENHANCED_MATCH_ARTIST_NAMES(name1 STRING, name2 STRING)
    RETURNS OBJECT
    LANGUAGE JAVASCRIPT
    AS
    $$
    
    /**
     * Normalize names to handle special cases:
     * 1. Remove/normalize generational suffixes
     * 2. Replace hyphens with spaces to handle hyphenated names
     * 3. Normalize special prefixes
     * 4. Remove bracket content
     */
    function normalizeNameForComparison(name) {
        if (!name) return "";
        
        // Remove content in brackets first
        name = name.replace(/\\([^\\)]*\\)/g, "");
        
        name = name.toUpperCase().trim();
        
        // 1. Remove generational suffixes: I, II, III, IV, V and their variations
        name = name.replace(/\\b(I{1,3}|IV|V|1ST|2ND|3RD|[4-9]TH)\\b$/g, "").trim();
        
        // 2. Replace hyphens with spaces for better matching
        name = name.replace(/-/g, " ");
        
        // 3. Special handling for prefixes - ensure consistent spacing
        const special_prefixes = ["VAN", "VON", "DE", "DER", "LA", "LE", "DI", "DEL", "DOS", "DA", "DU", "AL", "EL", "JESUS", "CHAVEZ"];
        
        // Create regex patterns for each prefix to match only when it's a standalone word
        for (const prefix of special_prefixes) {
            const regex = new RegExp(`\\\\b${prefix}\\\\b`, 'g');
            // Ensure consistent spacing for each prefix
            name = name.replace(regex, prefix);
        }
        
        // Remove extra whitespace
        name = name.replace(/\\s+/g, " ").trim();
        
        return name;
    }

    /**
     * Enhanced name parsing that handles multiple formats
     */
    function parseNameComponents(name) {
        if (!name || name === "") {
            return {formats: []};
        }

        // Normalize the name first
        name = normalizeNameForComparison(name);

        // Special suffixes and prefixes
        const suffixes = ["SENIOR", "SNR", "SR", "JUNIOR", "JNR", "JR"];
        const special_prefixes = ["VAN", "VON", "DE", "DER", "LA", "LE", "DI", "DEL", "DOS", "DA", "DU", "AL", "EL", "JESUS", "CHAVEZ"];

        // Clean and split the name
        let parts = name.split(/\\s+/);

        // Handle name with no spaces
        if (parts.length === 1) {
            return {formats: [{first: "", last: parts[0], middle: "", suffix: "", is_initial: false}]};
        }

        // Check for and extract suffixes
        let suffix = "";
        for (const suffixTerm of suffixes) {
            const suffixIndex = parts.findIndex(part => part === suffixTerm);
            if (suffixIndex !== -1) {
                suffix = parts[suffixIndex];
                parts.splice(suffixIndex, 1);
                break;
            }
        }

        // After removing suffix, check if we're back to a single name
        if (parts.length === 1) {
            return {formats: [{first: "", last: parts[0], middle: "", suffix: suffix, is_initial: false}]};
        }

        // Generate multiple format possibilities
        const formatsToCheck = [];
        
        // Check for special multi-word last names with prefixes like "VAN BEETHOVEN"
        let hasSpecialPrefix = false;
        let specialPrefixIndex = -1;
        
        for (let i = 0; i < parts.length; i++) {
            if (special_prefixes.includes(parts[i])) {
                hasSpecialPrefix = true;
                specialPrefixIndex = i;
                break;
            }
        }
        
        if (hasSpecialPrefix) {
            if (specialPrefixIndex === 0 && parts.length >= 2) {
                // Case: "VAN BEETHOVEN" or "VAN BEETHOVEN LUDWIG"
                if (parts.length === 2) {
                    // Just "VAN BEETHOVEN" - treat as compound last name
                    formatsToCheck.push({
                        first: "",
                        last: parts.join(" "),
                        middle: "",
                        suffix: suffix,
                        is_initial: false
                    });
                } else {
                    // "BEETHOVEN LUDWIG VAN" - LASTNAME FIRSTNAME PREFIX format
                    formatsToCheck.push({
                        first: parts[1],
                        last: parts[0] + " " + parts[2],
                        middle: parts.slice(3).join(" "),
                        suffix: suffix,
                        is_initial: parts[1] && parts[1].length === 1
                    });
                    
                    // "VAN BEETHOVEN LUDWIG" - PREFIX LASTNAME FIRSTNAME format
                    const lastName = parts.slice(0, 2).join(" ");
                    const firstName = parts[2];
                    const middleName = parts.slice(3).join(" ");
                    
                    formatsToCheck.push({
                        first: firstName,
                        last: lastName,
                        middle: middleName,
                        suffix: suffix,
                        is_initial: firstName && firstName.length === 1
                    });
                }
            } else if (specialPrefixIndex > 0) {
                // Case: "LUDWIG VAN BEETHOVEN" - FIRSTNAME PREFIX LASTNAME format
                const firstName = parts.slice(0, specialPrefixIndex).join(" ");
                const lastName = parts.slice(specialPrefixIndex).join(" ");
                
                formatsToCheck.push({
                    first: firstName,
                    last: lastName,
                    middle: "",
                    suffix: suffix,
                    is_initial: firstName.length === 1
                });
                
                // Also try reversed: LASTNAME FIRSTNAME format (VAN BEETHOVEN LUDWIG)
                formatsToCheck.push({
                    first: firstName,
                    last: lastName,
                    middle: "",
                    suffix: suffix,
                    is_initial: firstName.length === 1
                });
            }
        }
        
        // Standard format handling for all cases
        if (parts.length === 2) {
            // Two parts: ALWAYS try both orders for cases like "BERNIE BEN" vs "BEN BERNIE"
            formatsToCheck.push({
                first: parts[0],
                last: parts[1],
                middle: "",
                suffix: suffix,
                is_initial: parts[0].length === 1
            });
            
            formatsToCheck.push({
                first: parts[1],
                last: parts[0],
                middle: "",
                suffix: suffix,
                is_initial: parts[1].length === 1
            });
        } else if (parts.length >= 3) {
            // Format 1: LASTNAME FIRSTNAME MIDDLENAME (ADC Style)
            formatsToCheck.push({
                first: parts[1],
                last: parts[0],
                middle: parts.slice(2).join(" "),
                suffix: suffix,
                is_initial: parts[1].length === 1
            });
            
            // Format 2: FIRSTNAME MIDDLENAME LASTNAME (Muzooka Style)
            formatsToCheck.push({
                first: parts[0],
                last: parts[parts.length - 1],
                middle: parts.slice(1, parts.length - 1).join(" "),
                suffix: suffix,
                is_initial: parts[0].length === 1
            });
            
            // Format 3: FIRSTNAME LASTNAME (treating middle as part of first)
            formatsToCheck.push({
                first: parts.slice(0, parts.length - 1).join(" "),
                last: parts[parts.length - 1],
                middle: "",
                suffix: suffix,
                is_initial: false
            });
            
            // Format 4: LASTNAME FIRSTNAME (treating middle as part of last)
            formatsToCheck.push({
                first: parts[parts.length - 1],
                last: parts.slice(0, parts.length - 1).join(" "),
                middle: "",
                suffix: suffix,
                is_initial: parts[parts.length - 1].length === 1
            });
            
            // Format 5: For compound last names (first two parts as last name)
            if (parts.length >= 3) {
                formatsToCheck.push({
                    first: parts.slice(2).join(" "),
                    last: parts.slice(0, 2).join(" "),
                    middle: "",
                    suffix: suffix,
                    is_initial: false
                });
            }
            
            // Format 6: For compound last names (last two parts as last name)
            if (parts.length >= 3) {
                formatsToCheck.push({
                    first: parts.slice(0, parts.length - 2).join(" "),
                    last: parts.slice(parts.length - 2).join(" "),
                    middle: "",
                    suffix: suffix,
                    is_initial: false
                });
            }
        }

        return {formats: formatsToCheck};
    }

    /**
     * Calculate string similarity using Levenshtein distance
     */
    function calculateStringSimilarity(s1, s2) {
        if (!s1 || !s2) return 0;
        if (s1 === s2) return 1.0;

        const str1 = s1.toUpperCase().trim();
        const str2 = s2.toUpperCase().trim();

        if (str1 === str2) return 1.0;
        if (str1 === "" || str2 === "") return 0.0;

        const len1 = str1.length;
        const len2 = str2.length;
        const matrix = Array(len1 + 1).fill().map(() => Array(len2 + 1).fill(0));

        for (let i = 0; i <= len1; i++) matrix[i][0] = i;
        for (let j = 0; j <= len2; j++) matrix[0][j] = j;

        for (let i = 1; i <= len1; i++) {
            for (let j = 1; j <= len2; j++) {
                const cost = str1[i-1] === str2[j-1] ? 0 : 1;
                matrix[i][j] = Math.min(
                    matrix[i-1][j] + 1,
                    matrix[i][j-1] + 1,
                    matrix[i-1][j-1] + cost
                );
            }
        }

        const distance = matrix[len1][len2];
        const maxLen = Math.max(len1, len2);
        return 1 - (distance / maxLen);
    }

    /**
     * Jaro-Winkler similarity implementation
     */
    function jaroWinklerSimilarity(s1, s2) {
        if (!s1 || !s2 || s1.length === 0 || s2.length === 0) return 0;
        if (s1 === s2) return 1;
        
        let m = 0;
        let t = 0;
        let range = Math.floor(Math.max(s1.length, s2.length) / 2) - 1;
        range = Math.max(0, range);
        
        let s1Matches = new Array(s1.length).fill(false);
        let s2Matches = new Array(s2.length).fill(false);
        
        for (let i = 0; i < s1.length; i++) {
            let start = Math.max(0, i - range);
            let end = Math.min(i + range + 1, s2.length);
            
            for (let j = start; j < end; j++) {
                if (!s2Matches[j] && s1[i] === s2[j]) {
                    s1Matches[i] = true;
                    s2Matches[j] = true;
                    m++;
                    break;
                }
            }
        }
        
        if (m === 0) return 0;
        
        let k = 0;
        for (let i = 0; i < s1.length; i++) {
            if (s1Matches[i]) {
                while (k < s2.length && !s2Matches[k]) k++;
                if (k < s2.length && s1[i] !== s2[k]) t++;
                if (k < s2.length) k++;
            }
        }
        
        t = Math.floor(t / 2);
        let jaroSim = (m / s1.length + m / s2.length + (m - t) / m) / 3;
        
        let p = 0.1;
        let l = 0;
        
        for (let i = 0; i < Math.min(4, Math.min(s1.length, s2.length)); i++) {
            if (s1[i] === s2[i]) {
                l++;
            } else {
                break;
            }
        }
        
        return jaroSim + (l * p * (1 - jaroSim));
    }

    /**
     * Calculate match score between two name formats with enhanced logic
     */
    function calculateMatchScore(format1, format2) {
    if (!format1 || !format2) return 0;

    const first1 = format1.first || "";
    const last1 = format1.last || "";
    const middle1 = format1.middle || "";
    
    const first2 = format2.first || "";
    const last2 = format2.last || "";
    const middle2 = format2.middle || "";

    // STEP 1: Last name similarity (MUST match well for a good score)
    const lastNameScore = calculateStringSimilarity(last1, last2);
    
    // If last names don't match well, return a low score regardless of other factors
    if (lastNameScore < 0.8) {
        return Math.max(0, lastNameScore * 0.7); // Cap at 70% if last names don't match well
    }
    
    // STEP 2: First name similarity with enhanced initial handling
    // Only apply initial matching logic if last names match well
    let firstNameScore = 0;
    if (first1 && first2) {
        // Handle initials vs full names ONLY when last names match
        if (lastNameScore >= 0.9) { // Only if last names are very similar
            if ((first1.length === 1 && first2.length > 1) || (first2.length === 1 && first1.length > 1)) {
                const initial1 = first1.charAt(0);
                const initial2 = first2.charAt(0);
                firstNameScore = initial1 === initial2 ? 0.95 : 0; // High score for matching initials when last names match
            } else {
                firstNameScore = calculateStringSimilarity(first1, first2);
            }
        } else {
            // If last names don't match perfectly, require full first name match
            firstNameScore = calculateStringSimilarity(first1, first2);
        }
    } else if (!first1 && !first2) {
        firstNameScore = 1.0; // Both empty, perfect match
    }
    
    // STEP 3: Middle name handling - more lenient when last names match
    let middleNameScore = 1.0;
    if (middle1 && middle2) {
        middleNameScore = calculateStringSimilarity(middle1, middle2);
    } else if ((middle1 && !middle2) || (!middle1 && middle2)) {
        // If last names match well, be more forgiving of missing middle names
        if (lastNameScore >= 0.9) {
            middleNameScore = 0.9; // Less penalty when last names match
        } else {
            middleNameScore = 0.6; // More penalty when last names don't match well
        }
    }

    // STEP 4: Check for perfect match
    if (lastNameScore === 1.0 && firstNameScore === 1.0 && middleNameScore === 1.0) {
        return 1.0;
    }

    // STEP 5: Weighted score calculation with last name emphasis
    let totalScore = 0;
    
    if (lastNameScore >= 0.95) {
        // When last names match very well, use standard weighting
        totalScore = (lastNameScore * 0.60) + (firstNameScore * 0.35) + (middleNameScore * 0.05);
    } else if (lastNameScore >= 0.8) {
        // When last names match reasonably, increase last name importance
        totalScore = (lastNameScore * 0.75) + (firstNameScore * 0.20) + (middleNameScore * 0.05);
    } else {
        // When last names don't match well, heavily penalize
        totalScore = lastNameScore * 0.5; // Cap at 50% for poor last name matches
    }

    return Math.min(1.0, totalScore);
    }

    function extractCleanLastName(nameFormat) {
    if (!nameFormat || !nameFormat.last) return "";
    
    // Remove any remaining middle name components that might have leaked into last name
    let lastName = nameFormat.last.trim();
    
    // Handle compound last names with prefixes
    const special_prefixes = ["VAN", "VON", "DE", "DER", "LA", "LE", "DI", "DEL", "DOS", "DA", "DU", "AL", "EL"];
    
    // Don't split if it starts with a special prefix
    for (const prefix of special_prefixes) {
        if (lastName.startsWith(prefix + " ")) {
            return lastName; // Keep compound last name intact
        }
    }
    
    return lastName;
}

/**
 * Enhanced name matching with last name pre-filtering
 */
function enhancedMatchArtistNamesWithLastNameLogic(name1, name2) {
    try {
        if (!name1 || !name2 || name1.length === 0 || name2.length === 0) {
            return {
                jaro_winkler_score: 0.0,
                enhanced_score: 0.0,
                best_format_match: 0.0,
                normalized_match: 0.0,
                last_name_match: 0.0
            };
        }
        
        // Calculate original scores
        const jaroWinklerScore = jaroWinklerSimilarity(name1, name2);
        const normalizedName1 = normalizeNameForComparison(name1);
        const normalizedName2 = normalizeNameForComparison(name2);
        
        let normalizedMatch = 0.0;
        if (normalizedName1 === normalizedName2) {
            normalizedMatch = 1.0;
        } else {
            normalizedMatch = calculateStringSimilarity(normalizedName1, normalizedName2);
        }
        
        // Parse names into different formats
        const parsed1 = parseNameComponents(name1);
        const parsed2 = parseNameComponents(name2);
        
        // Find best format match with enhanced last name logic
        let bestFormatScore = 0;
        let bestLastNameMatch = 0;
        
        if (parsed1.formats && parsed2.formats) {
            for (const format1 of parsed1.formats) {
                for (const format2 of parsed2.formats) {
                    // Calculate last name similarity first
                    const lastNameSim = calculateStringSimilarity(
                        extractCleanLastName(format1), 
                        extractCleanLastName(format2)
                    );
                    
                    bestLastNameMatch = Math.max(bestLastNameMatch, lastNameSim);
                    
                    // Only proceed with full matching if last names have potential
                    if (lastNameSim >= 0.7) { // Threshold for considering a match
                        const currentScore = calculateMatchScore(format1, format2);
                        bestFormatScore = Math.max(bestFormatScore, currentScore);
                    }
                }
            }
        }
        
        // Calculate final enhanced score
        const enhancedScore = Math.max(
            jaroWinklerScore,
            normalizedMatch,
            bestFormatScore
        );
        
        return {
            jaro_winkler_score: jaroWinklerScore,
            enhanced_score: enhancedScore,
            best_format_match: bestFormatScore,
            normalized_match: normalizedMatch,
            last_name_match: bestLastNameMatch
        };
        
    } catch (e) {
        return {
            jaro_winkler_score: 0.0,
            enhanced_score: 0.0,
            best_format_match: 0.0,
            normalized_match: 0.0,
            last_name_match: 0.0
        };
    }
    }
    
    return enhancedMatchArtistNamesWithLastNameLogic(NAME1, NAME2);
    $$;
    """

    session.sql(create_udf_sql).collect()
    print("Enhanced JavaScript UDF for artist name matching created successfully.")


def create_part_by_part_match_table(session, work_ids=None):
    
    mode_description = "ALL WORKS"
    
    if work_ids:
        work_ids_str = "', '".join(work_ids)
        work_ids_formatted = f"'{work_ids_str}'"
        where_clause_step1 = f"WHERE APRA_WORK_ID IN ({work_ids_formatted})"
        where_clause_step2 = f"WHERE APRA_WORK_ID IN ({work_ids_formatted}) AND MUZOOKA_TRACK_ID IS NOT NULL"
        mode_description = f"SPECIFIC WORKS: {work_ids}"
    else:
        where_clause_step1 = ""  # No filtering for full mode
        where_clause_step2 = "WHERE MUZOOKA_TRACK_ID IS NOT NULL"  # Only filter out NULL tracks
    
    print(f"Creating part-by-part match table for {mode_description}...")
    print("Goal: Compare each ADC delimited part against each MZK delimited part")
    
    # Part-by-part matching query
    unified_sql = f"""
    WITH 
    -- Step 1: Get unique artist entries with their delimited parts
    unique_artist_names AS (
        SELECT DISTINCT
            APRA_WORK_ID,
            APRA_ARTIST_NAME,
            DELIMITED_PARTS AS ADC_DELIMITED_PARTS,
            MIN(APRA_ARTIST_ID) AS APRA_ARTIST_ID,
            MIN(RDC_WORKS_ID) AS RDC_WORKS_ID,
            MIN(APRA_ARTIST_CT) AS APRA_ARTIST_CT,
            MIN(APRA_TOTAL_ARTISTS) AS APRA_TOTAL_ARTISTS
        FROM ADC_ARTIST_DELIMITER_COUNT_CLEAN
        {where_clause_step1}
        GROUP BY APRA_WORK_ID, APRA_ARTIST_NAME, DELIMITED_PARTS
    ),
    
    -- Step 2: Get unique tracks per work
    unique_tracks AS (
        SELECT DISTINCT
            APRA_WORK_ID,
            MUZOOKA_TRACK_ID
        FROM ADC_ARTIST_DELIMITER_COUNT_CLEAN
        {where_clause_step2}
    ),
    
    -- Step 3: Get Muzooka track metadata with delimited parts
    track_metadata AS (
        SELECT DISTINCT
            MUZOOKA_TRACK_ID,
            DELIMITED_PARTS AS MZK_DELIMITED_PARTS,
            MZK_TOTAL_ARTISTS
        FROM MZK_ARTIST_DELIMITER_COUNT_CLEAN
    ),
    
    -- Step 4: Create base combinations (APRA artist x Track)
    base_combinations AS (
        SELECT 
            a.APRA_WORK_ID,
            a.APRA_ARTIST_ID,
            a.RDC_WORKS_ID,
            a.APRA_ARTIST_NAME,
            a.ADC_DELIMITED_PARTS,
            a.APRA_ARTIST_CT,
            a.APRA_TOTAL_ARTISTS,
            t.MUZOOKA_TRACK_ID,
            COALESCE(MAX(tm.MZK_TOTAL_ARTISTS), 0) AS MZK_TOTAL_ARTISTS
        FROM unique_artist_names a
        CROSS JOIN unique_tracks t
        LEFT JOIN track_metadata tm ON t.MUZOOKA_TRACK_ID = tm.MUZOOKA_TRACK_ID
        WHERE a.APRA_WORK_ID = t.APRA_WORK_ID
        GROUP BY a.APRA_WORK_ID, a.APRA_ARTIST_ID, a.RDC_WORKS_ID, a.APRA_ARTIST_NAME, 
                 a.ADC_DELIMITED_PARTS, a.APRA_ARTIST_CT, a.APRA_TOTAL_ARTISTS, t.MUZOOKA_TRACK_ID
    ),
    
    -- Step 5: Explode ADC delimited parts into individual parts
    adc_parts_exploded AS (
        SELECT 
            bc.*,
            adc_part.VALUE::STRING AS ADC_INDIVIDUAL_PART,
            adc_part.INDEX AS ADC_PART_INDEX
        FROM base_combinations bc,
        LATERAL FLATTEN(PARSE_JSON(bc.ADC_DELIMITED_PARTS)) adc_part
    ),
    
    -- Step 6: Get MZK delimited parts for each track and explode them
    mzk_parts_exploded AS (
        SELECT 
            tm.MUZOOKA_TRACK_ID,
            tm.MZK_DELIMITED_PARTS,
            mzk_part.VALUE::STRING AS MZK_INDIVIDUAL_PART,
            mzk_part.INDEX AS MZK_PART_INDEX
        FROM track_metadata tm,
        LATERAL FLATTEN(PARSE_JSON(tm.MZK_DELIMITED_PARTS)) mzk_part
    ),
    
    -- Step 7: Create all combinations of ADC parts x MZK parts for each track
    part_combinations AS (
        SELECT 
            ape.*,
            mpe.MZK_DELIMITED_PARTS,
            mpe.MZK_INDIVIDUAL_PART,
            mpe.MZK_PART_INDEX,
            -- Calculate match score between individual parts
            ENHANCED_MATCH_ARTIST_NAMES(ape.ADC_INDIVIDUAL_PART, mpe.MZK_INDIVIDUAL_PART):enhanced_score::FLOAT AS PART_MATCH_SCORE
        FROM adc_parts_exploded ape
        LEFT JOIN mzk_parts_exploded mpe 
        ON ape.MUZOOKA_TRACK_ID = mpe.MUZOOKA_TRACK_ID
    ),
    
    -- Step 8: Find best matching MZK part for each ADC part
    best_part_matches AS (
        SELECT 
            APRA_WORK_ID,
            APRA_ARTIST_ID,
            MUZOOKA_TRACK_ID,
            APRA_ARTIST_NAME,
            ADC_INDIVIDUAL_PART,
            ADC_PART_INDEX,
            MAX(PART_MATCH_SCORE) AS BEST_PART_SCORE
        FROM part_combinations
        GROUP BY APRA_WORK_ID, APRA_ARTIST_ID, MUZOOKA_TRACK_ID, APRA_ARTIST_NAME, ADC_INDIVIDUAL_PART, ADC_PART_INDEX
    ),
    
    -- Step 9: Get the MZK delimited parts for the track that had the best overall matching
    best_track_mzk_parts AS (
        SELECT 
            pc.APRA_WORK_ID,
            pc.APRA_ARTIST_ID,
            pc.MUZOOKA_TRACK_ID,
            pc.APRA_ARTIST_NAME,
            pc.MZK_DELIMITED_PARTS,
            SUM(bpm.BEST_PART_SCORE) AS TOTAL_PART_MATCH_SCORE,
            COUNT(DISTINCT pc.ADC_PART_INDEX) AS ADC_PARTS_COUNT
        FROM part_combinations pc
        INNER JOIN best_part_matches bpm 
        ON pc.APRA_WORK_ID = bpm.APRA_WORK_ID 
        AND pc.APRA_ARTIST_ID = bpm.APRA_ARTIST_ID 
        AND pc.MUZOOKA_TRACK_ID = bpm.MUZOOKA_TRACK_ID
        AND pc.ADC_INDIVIDUAL_PART = bpm.ADC_INDIVIDUAL_PART
        AND pc.PART_MATCH_SCORE = bpm.BEST_PART_SCORE
        GROUP BY pc.APRA_WORK_ID, pc.APRA_ARTIST_ID, pc.MUZOOKA_TRACK_ID, pc.APRA_ARTIST_NAME, pc.MZK_DELIMITED_PARTS
    ),
    
    -- Step 10: Select the best MZK delimited parts per artist-track combination
    final_best_matches AS (
        SELECT 
            *,
            ROW_NUMBER() OVER (
                PARTITION BY APRA_WORK_ID, APRA_ARTIST_ID, MUZOOKA_TRACK_ID 
                ORDER BY TOTAL_PART_MATCH_SCORE DESC, MZK_DELIMITED_PARTS
            ) as rn
        FROM best_track_mzk_parts
    ),
    
    -- Step 11: Calculate artist match percentage at work-track level
    -- New logic: SUM(name_match_scores >= 0.8) / MAX(apra_total_artists, mzk_total_artists)
    work_track_artist_percentage AS (
        SELECT 
            bc.APRA_WORK_ID,
            bc.MUZOOKA_TRACK_ID,
            -- Count artists with name_match_score >= 0.8
            SUM(CASE 
                WHEN CASE 
                    WHEN bc.APRA_ARTIST_CT > 0 THEN
                        COALESCE(fbm.TOTAL_PART_MATCH_SCORE, 0) / bc.APRA_ARTIST_CT
                    ELSE 0.0
                END >= 0.8 
                THEN 1.0 
                ELSE 0.0 
            END) AS high_match_artist_count,
            -- Get the maximum of APRA and MZK total artists
            GREATEST(MAX(bc.APRA_TOTAL_ARTISTS), MAX(bc.MZK_TOTAL_ARTISTS)) AS max_total_artists,
            -- Calculate the artist match percentage
            CASE 
                WHEN GREATEST(MAX(bc.APRA_TOTAL_ARTISTS), MAX(bc.MZK_TOTAL_ARTISTS)) > 0 THEN
                    SUM(CASE 
                        WHEN CASE 
                            WHEN bc.APRA_ARTIST_CT > 0 THEN
                                COALESCE(fbm.TOTAL_PART_MATCH_SCORE, 0) / bc.APRA_ARTIST_CT
                            ELSE 0.0
                        END >= 0.8 
                        THEN 1.0 
                        ELSE 0.0 
                    END) / GREATEST(MAX(bc.APRA_TOTAL_ARTISTS), MAX(bc.MZK_TOTAL_ARTISTS))
                ELSE 0.0
            END AS ARTIST_MTCH_PCNTG
        FROM base_combinations bc
        LEFT JOIN final_best_matches fbm
            ON bc.APRA_WORK_ID = fbm.APRA_WORK_ID 
            AND bc.APRA_ARTIST_ID = fbm.APRA_ARTIST_ID 
            AND bc.MUZOOKA_TRACK_ID = fbm.MUZOOKA_TRACK_ID
            AND fbm.rn = 1
        GROUP BY bc.APRA_WORK_ID, bc.MUZOOKA_TRACK_ID
    )
    
    -- Step 12: Final result set with part-by-part matching logic
    SELECT
        bc.RDC_WORKS_ID,
        bc.APRA_ARTIST_ID,
        bc.APRA_WORK_ID,
        bc.MUZOOKA_TRACK_ID,
        bc.APRA_ARTIST_NAME,
        --bc.ADC_DELIMITED_PARTS,
        --COALESCE(fbm.MZK_DELIMITED_PARTS, '[]') AS MZK_DELIMITED_PARTS,
        bc.APRA_TOTAL_ARTISTS,
        bc.MZK_TOTAL_ARTISTS,
        bc.APRA_ARTIST_CT,
        COALESCE(fbm.TOTAL_PART_MATCH_SCORE, 0) AS PART_MATCH_SCORE,
        CASE 
            WHEN bc.APRA_ARTIST_CT > 0 THEN
                COALESCE(fbm.TOTAL_PART_MATCH_SCORE, 0) / bc.APRA_ARTIST_CT
            ELSE 0.0
        END AS NAME_MATCH_SCORE,
        ROUND(COALESCE(wtap.ARTIST_MTCH_PCNTG, 0), 6) AS ARTIST_MTCH_PCNTG
    FROM 
        base_combinations bc
    LEFT JOIN 
        final_best_matches fbm
    ON bc.APRA_WORK_ID = fbm.APRA_WORK_ID 
       AND bc.APRA_ARTIST_ID = fbm.APRA_ARTIST_ID 
       AND bc.MUZOOKA_TRACK_ID = fbm.MUZOOKA_TRACK_ID
       AND fbm.rn = 1
    LEFT JOIN 
        work_track_artist_percentage wtap
    ON bc.APRA_WORK_ID = wtap.APRA_WORK_ID 
       AND bc.MUZOOKA_TRACK_ID = wtap.MUZOOKA_TRACK_ID
    ORDER BY
        bc.APRA_WORK_ID, bc.APRA_ARTIST_NAME, bc.MUZOOKA_TRACK_ID
    """
    
    # Execute and create the results table
    results = session.sql(unified_sql)
    
    # Choose table name based on mode
    table_name = "ARTIST_PART_BY_PART_MATCH_FULL" if not work_ids else "ARTIST_PART_BY_PART_MATCH_SPECIFIC"
    
    results.write.mode("overwrite").save_as_table(table_name)
    
    print(f"✅ Table {table_name} created successfully!")
    print(f"Processing mode: {mode_description}")
    
    return table_name


def run_specific_works_pipeline(session, work_ids):
    """Run pipeline for specific work IDs using part-by-part matching"""
    
    if not work_ids:
        print("No work IDs provided.")
        return
    
    print(f"Running PART-BY-PART artist matching for work IDs: {work_ids}")
    
    # Create UDF first
    create_enhanced_udf(session)
    
    # Run part-by-part pipeline for specific work IDs
    table_name = create_part_by_part_match_table(session, work_ids)
    
    # Show detailed results for specific works if requested
    if 'GW39809338' in work_ids:
        print(f"\n=== DETAILED PART-BY-PART RESULTS FOR GW39809338 ===")
        detailed_sql = f"""
        SELECT 
            APRA_WORK_ID,
            MUZOOKA_TRACK_ID,
            APRA_ARTIST_NAME,
            ADC_DELIMITED_PARTS,
            MZK_DELIMITED_PARTS,
            APRA_ARTIST_CT,
            PART_MATCH_SCORE,
            NAME_MATCH_SCORE,
            ARTIST_MTCH_PCNTG
        FROM {table_name}
        WHERE APRA_WORK_ID = 'GW39809338'
        ORDER BY MUZOOKA_TRACK_ID, APRA_ARTIST_NAME
        """
        
        detailed_results = session.sql(detailed_sql)
        detailed_results.show()
        
        # Show part-by-part breakdown
        print(f"\n=== PART-BY-PART MATCHING BREAKDOWN FOR GW39809338 ===")
        breakdown_sql = f"""
        WITH part_details AS (
            SELECT 
                APRA_ARTIST_NAME,
                ADC_DELIMITED_PARTS,
                MZK_DELIMITED_PARTS,
                PART_MATCH_SCORE,
                NAME_MATCH_SCORE,
                APRA_ARTIST_CT
            FROM {table_name}
            WHERE APRA_WORK_ID = 'GW39809338'
        )
        SELECT 
            APRA_ARTIST_NAME,
            ADC_DELIMITED_PARTS as "ADC Parts",
            MZK_DELIMITED_PARTS as "MZK Parts", 
            APRA_ARTIST_CT as "ADC Part Count",
            PART_MATCH_SCORE as "Total Part Score",
            NAME_MATCH_SCORE as "Normalized Score (Score/Count)"
        FROM part_details
        ORDER BY NAME_MATCH_SCORE DESC
        """
        
        breakdown_results = session.sql(breakdown_sql)
        breakdown_results.show()
    
    print(f"\n🎯 PART-BY-PART PIPELINE COMPLETED FOR SPECIFIC WORKS!")
    print(f"📊 Table created: {table_name}")
    

def run_full_pipeline(session):
    """Run the complete pipeline for all works using part-by-part matching"""
    
    print("Starting PART-BY-PART artist name matching pipeline for ALL WORKS...")
    
    # Create UDF first
    create_enhanced_udf(session)
    
    # Run part-by-part pipeline for all works
    table_name = create_part_by_part_match_table(session, work_ids=None)
    
    print(f"\n🎯 PART-BY-PART PIPELINE COMPLETED FOR ALL WORKS!")
    print(f"📊 Table created: {table_name}")
  

def main(session):
    """
    Main function with part-by-part matching implementation
    """
    
    # Set run mode and parameters
    run_mode = 'full'  # Options: 'full', 'specific_works'
    
    # Define specific work IDs for testing
    specific_work_ids = [
        'GW39809338',      # The work mentioned in the question
        #'GW71382587',
        'GW01234636'
    ]
    
    if run_mode.lower() == 'full':
        # Run part-by-part pipeline for all data
        run_full_pipeline(session)
        
    elif run_mode.lower() == 'specific_works':
        # Run part-by-part pipeline for specific work IDs
        run_specific_works_pipeline(session, specific_work_ids)
        
    else:
        print(f"Invalid run mode: {run_mode}. Please specify 'full' or 'specific_works'")
        return
    
    print(f"\n🎯 PART-BY-PART ARTIST NAME MATCHING COMPLETED!")
    print(f"Mode: {run_mode}")
   
    if run_mode.lower() == 'specific_works':
        print(f"Processed work IDs: {specific_work_ids}")


if __name__ == "__main__":
    main(session)